<a href="https://colab.research.google.com/github/amzad-786githumb/AIR_LLM_Research/blob/main/05_Dataset_and_Feature_Profiling.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# ============================================================
# NOTEBOOK 05.0 — ENVIRONMENT AND IMPORTS
# ============================================================

print("=" * 100)
print("AIR-LLM — NOTEBOOK 05")
print("LLM RECOMMENDATION ENGINE")
print("=" * 100)

from pathlib import Path
import json
import hashlib
import random
import warnings

import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")

SEED = 42

random.seed(SEED)
np.random.seed(SEED)

print("\nEnvironment initialized")
print(f"Random seed : {SEED}")

AIR-LLM — NOTEBOOK 05
LLM RECOMMENDATION ENGINE

Environment initialized
Random seed : 42


In [2]:
# ============================================================
# NOTEBOOK 05.1 — PROJECT CONFIGURATION
# ============================================================

print("=" * 100)
print("AIR-LLM — NOTEBOOK 05.1")
print("LOADING PROJECT CONFIGURATION")
print("=" * 100)

from google.colab import drive

drive.mount(
    "/content/drive",
    force_remount=False
)

PROJECT_ROOT = Path(
    "/content/drive/MyDrive/AIR_LLM_Research"
)

if not PROJECT_ROOT.is_dir():
    raise FileNotFoundError(
        f"AIR-LLM project directory not found:\n"
        f"{PROJECT_ROOT}"
    )

DATA_DIR = PROJECT_ROOT / "data"

NOTEBOOK_03_DIR = DATA_DIR / "notebook_03"
NOTEBOOK_04_DIR = DATA_DIR / "notebook_04"
NOTEBOOK_05_DIR = DATA_DIR / "notebook_05"

RECOMMENDATION_DIR = NOTEBOOK_05_DIR / "recommendations"
CONTEXT_DIR = NOTEBOOK_05_DIR / "contexts"
VALIDATION_DIR = NOTEBOOK_05_DIR / "validation"
METADATA_DIR = NOTEBOOK_05_DIR / "metadata"
DIAGNOSTIC_DIR = NOTEBOOK_05_DIR / "diagnostics"

for directory in [
    NOTEBOOK_05_DIR,
    RECOMMENDATION_DIR,
    CONTEXT_DIR,
    VALIDATION_DIR,
    METADATA_DIR,
    DIAGNOSTIC_DIR
]:
    directory.mkdir(
        parents=True,
        exist_ok=True
    )

# ------------------------------------------------------------
# All research datasets
# ------------------------------------------------------------

DATASETS = [
    "adult_income",
    "bank_marketing",
    "diabetes_130us"
]

TARGET_REGISTRY = {
    "adult_income": "income",
    "bank_marketing": "y",
    "diabetes_130us": "readmitted"
}

# ------------------------------------------------------------
# Datasets participating in the LLM recommendation experiment
# ------------------------------------------------------------

RECOMMENDATION_DATASETS = [
    "adult_income",
    "diabetes_130us"
]

# ------------------------------------------------------------
# Technical features excluded from recommendations
# ------------------------------------------------------------

TECHNICAL_FEATURES = {
    "__air_llm_row_id"
}

# ------------------------------------------------------------
# Recommendation configuration
# ------------------------------------------------------------

TOP_K = 5

RECOMMENDATION_VERSION = "AIR-LLM-1.1"

RECOMMENDATION_SOURCE = (
    "LLM_guided_structured_recommendation"
)

# ------------------------------------------------------------
# Configuration validation
# ------------------------------------------------------------

missing_targets = [
    dataset_id
    for dataset_id in DATASETS
    if dataset_id not in TARGET_REGISTRY
]

if missing_targets:
    raise RuntimeError(
        "Missing target registry entries:\n"
        + "\n".join(
            f"  - {dataset_id}"
            for dataset_id in missing_targets
        )
    )

invalid_recommendation_datasets = [
    dataset_id
    for dataset_id in RECOMMENDATION_DATASETS
    if dataset_id not in DATASETS
]

if invalid_recommendation_datasets:
    raise RuntimeError(
        "Invalid recommendation dataset(s):\n"
        + "\n".join(
            f"  - {dataset_id}"
            for dataset_id in invalid_recommendation_datasets
        )
    )

if TOP_K <= 0:
    raise RuntimeError(
        "TOP_K must be positive."
    )

print("\nPROJECT")
print("-" * 100)
print(f"Root : {PROJECT_ROOT}")

print("\nALL RESEARCH DATASETS")
print("-" * 100)

for dataset_id in DATASETS:
    print(
        f"{dataset_id:<20}"
        f"| target = {TARGET_REGISTRY[dataset_id]}"
    )

print("\nRECOMMENDATION DATASETS")
print("-" * 100)

for dataset_id in RECOMMENDATION_DATASETS:
    print(
        f"{dataset_id:<20}"
        f"| target = {TARGET_REGISTRY[dataset_id]}"
    )

print("\nRECOMMENDATION CONFIGURATION")
print("-" * 100)
print(f"TOP-K                 : {TOP_K}")
print(f"Version               : {RECOMMENDATION_VERSION}")
print(f"Source                : {RECOMMENDATION_SOURCE}")

print("\nNotebook 05 directories created successfully.")

AIR-LLM — NOTEBOOK 05.1
LOADING PROJECT CONFIGURATION
Mounted at /content/drive

PROJECT
----------------------------------------------------------------------------------------------------
Root : /content/drive/MyDrive/AIR_LLM_Research

ALL RESEARCH DATASETS
----------------------------------------------------------------------------------------------------
adult_income        | target = income
bank_marketing      | target = y
diabetes_130us      | target = readmitted

RECOMMENDATION DATASETS
----------------------------------------------------------------------------------------------------
adult_income        | target = income
diabetes_130us      | target = readmitted

RECOMMENDATION CONFIGURATION
----------------------------------------------------------------------------------------------------
TOP-K                 : 5
Version               : AIR-LLM-1.1
Source                : LLM_guided_structured_recommendation

Notebook 05 directories created successfully.


In [3]:
# ============================================================
# NOTEBOOK 05.2 — LOAD TRAINING DATA
# ============================================================

print("=" * 100)
print("AIR-LLM — NOTEBOOK 05.2")
print("LOADING TRAINING DATA")
print("=" * 100)


TRAINING_DATA = {}


def locate_processed_dataset(dataset_id):

    candidates = [

        DATA_DIR
        / "processed"
        / f"{dataset_id}_processed.csv",

        DATA_DIR
        / "notebook_02"
        / "processed"
        / f"{dataset_id}_processed.csv",

        DATA_DIR
        / "notebook_02"
        / "datasets"
        / f"{dataset_id}_processed.csv",

        PROJECT_ROOT
        / f"{dataset_id}_processed.csv"
    ]

    for path in candidates:

        if path.exists():

            return path

    return None


for dataset_id in DATASETS:

    path = locate_processed_dataset(
        dataset_id
    )

    if path is None:

        raise FileNotFoundError(
            f"Processed dataset not found for "
            f"{dataset_id}.\n"
            f"Expected one of:\n"
            + "\n".join(
                str(p)
                for p in [
                    DATA_DIR
                    / "processed"
                    / f"{dataset_id}_processed.csv",
                    DATA_DIR
                    / "notebook_02"
                    / "processed"
                    / f"{dataset_id}_processed.csv"
                ]
            )
        )

    df = pd.read_csv(
        path
    )

    target = TARGET_REGISTRY[
        dataset_id
    ]

    if target not in df.columns:

        raise KeyError(
            f"Target '{target}' is missing "
            f"from {dataset_id}"
        )

    TRAINING_DATA[
        dataset_id
    ] = df

    print(
        f"{dataset_id:<20} | "
        f"Rows: {len(df):>7,} | "
        f"Columns: {len(df.columns):>3} | "
        f"Target: {target}"
    )


print(
    "\nTraining datasets loaded:",
    len(TRAINING_DATA)
)

AIR-LLM — NOTEBOOK 05.2
LOADING TRAINING DATA
adult_income         | Rows:  32,561 | Columns:  15 | Target: income
bank_marketing       | Rows:  45,211 | Columns:  17 | Target: y
diabetes_130us       | Rows: 101,766 | Columns:  48 | Target: readmitted

Training datasets loaded: 3


In [4]:
# ============================================================
# NOTEBOOK 05.3 — LOAD AND NORMALIZE NOTEBOOK 04 FEATURE PROFILE
# ============================================================

print("=" * 100)
print("AIR-LLM — NOTEBOOK 05.3")
print("LOADING AND NORMALIZING NOTEBOOK 04 FEATURE PROFILE")
print("=" * 100)

# ============================================================
# 1. PATH AND PROFILE
# ============================================================

FEATURE_PROFILE_PATH = (
    PROJECT_ROOT
    / "data"
    / "notebook_04"
    / "profiles"
    / "feature"
    / "feature_profile.csv"
)

if not FEATURE_PROFILE_PATH.exists():
    raise FileNotFoundError(
        f"Notebook 04 feature profile not found:\n{FEATURE_PROFILE_PATH}"
    )

FEATURE_PROFILE_DF = pd.read_csv(FEATURE_PROFILE_PATH)

FEATURE_PROFILE_DF.columns = (
    FEATURE_PROFILE_DF.columns.astype(str).str.strip()
)

print("\nNOTEBOOK 04 FEATURE PROFILE")
print("-" * 100)
print(f"Rows    : {len(FEATURE_PROFILE_DF):,}")
print(f"Columns : {len(FEATURE_PROFILE_DF.columns)}")

# ============================================================
# 2. REQUIRED COLUMNS
# ============================================================

REQUIRED_COLUMNS = [
    "dataset_id",
    "feature",
    "missing_rate"
]

missing_columns = [
    c for c in REQUIRED_COLUMNS
    if c not in FEATURE_PROFILE_DF.columns
]

if missing_columns:
    raise RuntimeError(
        "Missing required columns:\n"
        + "\n".join(f"  - {c}" for c in missing_columns)
    )

# ============================================================
# 3. NORMALIZATION
# ============================================================

FEATURE_PROFILE_DF["dataset_id"] = (
    FEATURE_PROFILE_DF["dataset_id"]
    .astype(str)
    .str.strip()
)

FEATURE_PROFILE_DF["feature"] = (
    FEATURE_PROFILE_DF["feature"]
    .astype(str)
    .str.strip()
)

FEATURE_PROFILE_DF["missing_rate"] = pd.to_numeric(
    FEATURE_PROFILE_DF["missing_rate"],
    errors="coerce"
)

if FEATURE_PROFILE_DF["missing_rate"].isna().any():
    raise RuntimeError("Invalid missing_rate values detected.")

# ============================================================
# 4. TECHNICAL FEATURES
# ============================================================

TECHNICAL_FEATURES = {
    "__air_llm_row_id",
    "_air_llm_row_id",
    "row_id",
    "index"
}

PROFILE_DF = FEATURE_PROFILE_DF[
    ~FEATURE_PROFILE_DF["feature"].isin(TECHNICAL_FEATURES)
].copy()

# ============================================================
# 5. REMOVE TARGET VARIABLES
# ============================================================

TARGET_ROWS = []

for dataset_id, target in TARGET_REGISTRY.items():

    mask = (
        (PROFILE_DF["dataset_id"] == dataset_id)
        &
        (
            PROFILE_DF["feature"].str.lower()
            == str(target).lower()
        )
    )

    if mask.any():
        TARGET_ROWS.extend(
            PROFILE_DF.loc[mask, "feature"].tolist()
        )

    PROFILE_DF = PROFILE_DF.loc[~mask].copy()

# ============================================================
# 6. FEATURE TYPE
# ============================================================

def recover_feature_type(row):

    if "feature_type" in row.index:

        value = row["feature_type"]

        if pd.notna(value):

            value = str(value).strip().lower()

            if value not in {"", "unknown", "none", "nan"}:

                if any(
                    x in value
                    for x in ["categor", "object", "string", "bool"]
                ):
                    return "categorical"

                if any(
                    x in value
                    for x in ["numeric", "float", "int", "double"]
                ):
                    return "numerical"

    dtype = str(row.get("dtype", "")).lower()

    if any(
        x in dtype
        for x in ["object", "category", "string", "bool"]
    ):
        return "categorical"

    if any(
        x in dtype
        for x in ["int", "float", "double", "numeric"]
    ):
        return "numerical"

    if pd.notna(row.get("mean", np.nan)):
        return "numerical"

    return "categorical"


PROFILE_DF["feature_type"] = (
    PROFILE_DF.apply(recover_feature_type, axis=1)
)

# ============================================================
# 7. INCOMPLETE FEATURES ONLY
# ============================================================

CONTEXT_DF = PROFILE_DF[
    PROFILE_DF["missing_rate"] > 0
].copy()

CONTEXT_DF["target"] = (
    CONTEXT_DF["dataset_id"].map(TARGET_REGISTRY)
)

CONTEXT_DF = (
    CONTEXT_DF
    .sort_values(["dataset_id", "feature"])
    .reset_index(drop=True)
)

# ============================================================
# 8. EXPECTED NOTEBOOK 04 POPULATION
# ============================================================

EXPECTED_FEATURES = {
    ("adult_income", "workclass"),
    ("adult_income", "occupation"),
    ("adult_income", "native_country"),

    ("diabetes_130us", "race"),
    ("diabetes_130us", "weight"),
    ("diabetes_130us", "payer_code"),
    ("diabetes_130us", "medical_specialty"),
    ("diabetes_130us", "diag_1"),
    ("diabetes_130us", "diag_2"),
    ("diabetes_130us", "diag_3"),
    ("diabetes_130us", "max_glu_serum"),
    ("diabetes_130us", "a1cresult")
}

ACTUAL_FEATURES = set(
    zip(
        CONTEXT_DF["dataset_id"],
        CONTEXT_DF["feature"]
    )
)

missing_features = EXPECTED_FEATURES - ACTUAL_FEATURES
unexpected_features = ACTUAL_FEATURES - EXPECTED_FEATURES

if missing_features:
    raise RuntimeError(
        "Missing expected Notebook 04 features:\n"
        + "\n".join(
            f"{d} | {f}"
            for d, f in sorted(missing_features)
        )
    )

if unexpected_features:
    raise RuntimeError(
        "Unexpected recommendation features:\n"
        + "\n".join(
            f"{d} | {f}"
            for d, f in sorted(unexpected_features)
        )
    )

# ============================================================
# 9. VALIDATION
# ============================================================

if len(CONTEXT_DF) != 12:
    raise RuntimeError(
        f"Expected 12 incomplete features, found {len(CONTEXT_DF)}."
    )

if CONTEXT_DF.duplicated(
    subset=["dataset_id", "feature"]
).any():
    raise RuntimeError("Duplicate dataset-feature contexts detected.")

if CONTEXT_DF["feature_type"].isin(
    ["categorical", "numerical"]
).all() is False:
    raise RuntimeError("Invalid feature types detected.")

if not (
    CONTEXT_DF["missing_rate"] > 0
).all():
    raise RuntimeError("Complete features entered the recommendation population.")

# ============================================================
# 10. EXPLICIT HANDOFF TO NOTEBOOK 05.4
# ============================================================

RECOMMENDATION_PROFILE_DF = CONTEXT_DF.copy()

# ============================================================
# 11. OUTPUT
# ============================================================

print("\nRECOMMENDATION POPULATION")
print("-" * 100)

for dataset_id in TARGET_REGISTRY:

    count = len(
        RECOMMENDATION_PROFILE_DF[
            RECOMMENDATION_PROFILE_DF["dataset_id"] == dataset_id
        ]
    )

    print(
        f"{dataset_id:<20} | "
        f"incomplete features = {count}"
    )

print("\nVALIDATION")
print("-" * 100)
print("Target exclusion          : PASSED")
print("Technical exclusion       : PASSED")
print("Semantic feature typing   : PASSED")
print("Feature-level consistency : PASSED")
print("Duplicate check           : PASSED")
print("Population size           : 12/12 PASSED")

print("\nHandoff")
print("-" * 100)
print(
    f"RECOMMENDATION_PROFILE_DF : "
    f"{len(RECOMMENDATION_PROFILE_DF)} rows"
)

print("\n" + "=" * 100)
print("NOTEBOOK 05.3 READY FOR NOTEBOOK 05.4")
print("=" * 100)

AIR-LLM — NOTEBOOK 05.3
LOADING AND NORMALIZING NOTEBOOK 04 FEATURE PROFILE

NOTEBOOK 04 FEATURE PROFILE
----------------------------------------------------------------------------------------------------
Rows    : 80
Columns : 33

RECOMMENDATION POPULATION
----------------------------------------------------------------------------------------------------
adult_income         | incomplete features = 3
bank_marketing       | incomplete features = 0
diabetes_130us       | incomplete features = 9

VALIDATION
----------------------------------------------------------------------------------------------------
Target exclusion          : PASSED
Technical exclusion       : PASSED
Semantic feature typing   : PASSED
Feature-level consistency : PASSED
Duplicate check           : PASSED
Population size           : 12/12 PASSED

Handoff
----------------------------------------------------------------------------------------------------
RECOMMENDATION_PROFILE_DF : 12 rows

NOTEBOOK 05.3 READY FOR

In [5]:
# ============================================================
# NOTEBOOK 05.4 — BUILD STRUCTURED LLM CONTEXT
# RESEARCH-GRADE FINAL VERSION
# ============================================================

print("=" * 100)
print("AIR-LLM — NOTEBOOK 05.4")
print("BUILDING RESEARCH-GRADE LLM CONTEXT")
print("=" * 100)

import numpy as np
import pandas as pd


# ============================================================
# 1. REQUIRED INPUT VALIDATION
# ============================================================

REQUIRED_PROFILE_COLUMNS = [
    "dataset_id",
    "feature",
    "feature_type",
    "missing_rate"
]

missing_profile_columns = [
    c for c in REQUIRED_PROFILE_COLUMNS
    if c not in RECOMMENDATION_PROFILE_DF.columns
]

if missing_profile_columns:
    raise RuntimeError(
        "Notebook 05.3 recommendation profile is missing "
        "required columns:\n"
        + "\n".join(
            f"  - {c}" for c in missing_profile_columns
        )
    )

if "TRAINING_DATA" not in globals():
    raise RuntimeError(
        "TRAINING_DATA is not available."
    )

if "TARGET_REGISTRY" not in globals():
    raise RuntimeError(
        "TARGET_REGISTRY is not available."
    )


# ============================================================
# 2. SAFE VALUE FUNCTIONS
# ============================================================

def get_value(row, column, default=np.nan):

    if column not in row.index:
        return default

    value = row[column]

    if pd.isna(value):
        return default

    return value


def safe_float(value, default=np.nan):

    try:

        if pd.isna(value):
            return default

        value = float(value)

        if not np.isfinite(value):
            return default

        return value

    except Exception:

        return default


def safe_int(value, default=0):

    try:

        if pd.isna(value):
            return default

        return int(value)

    except Exception:

        return default


# ============================================================
# 3. BUILD CONTEXTS
# ============================================================

CONTEXT_ROWS = []
REPRESENTATION_WARNINGS = []

for _, row in RECOMMENDATION_PROFILE_DF.iterrows():

    dataset_id = str(
        row["dataset_id"]
    ).strip()

    feature = str(
        row["feature"]
    ).strip()

    feature_type = (
        str(row["feature_type"])
        .strip()
        .lower()
    )

    target = TARGET_REGISTRY.get(
        dataset_id
    )


    # --------------------------------------------------------
    # Dataset representation
    # --------------------------------------------------------

    df = TRAINING_DATA.get(
        dataset_id
    )

    if df is None:

        REPRESENTATION_WARNINGS.append({
            "dataset_id": dataset_id,
            "feature": feature,
            "reason":
                "Dataset is present in Notebook 04 profile "
                "but unavailable in TRAINING_DATA."
        })


    feature_present = (
        df is not None
        and feature in df.columns
    )


    if not feature_present:

        REPRESENTATION_WARNINGS.append({
            "dataset_id": dataset_id,
            "feature": feature,
            "reason":
                "Feature exists in Notebook 04 profile "
                "but is absent from TRAINING_DATA."
        })


    # --------------------------------------------------------
    # Dataset-level statistics
    # --------------------------------------------------------

    if df is not None:

        dataset_rows = len(df)

        dataset_columns = len(
            df.columns
        )

        dataset_missing_rate = float(
            df.isna()
            .mean()
            .mean()
        )

    else:

        dataset_rows = 0
        dataset_columns = 0
        dataset_missing_rate = np.nan


    # --------------------------------------------------------
    # Feature profile statistics
    # --------------------------------------------------------

    feature_cardinality = safe_int(
        get_value(
            row,
            "unique_count",
            0
        ),
        0
    )


    mean_value = safe_float(
        get_value(
            row,
            "mean",
            np.nan
        )
    )


    variance_value = safe_float(
        get_value(
            row,
            "variance",
            np.nan
        )
    )


    skewness_value = safe_float(
        get_value(
            row,
            "skewness",
            np.nan
        )
    )


    outlier_rate_value = safe_float(
        get_value(
            row,
            "outlier_rate",
            np.nan
        )
    )


    entropy_value = safe_float(
        get_value(
            row,
            "entropy",
            np.nan
        )
    )


    mutual_information_value = safe_float(
        get_value(
            row,
            "mutual_information",
            np.nan
        )
    )


    target_relevance_value = safe_float(
        get_value(
            row,
            "target_relevance_primary",
            0.0
        ),
        0.0
    )


    target_relevance_type = str(
        get_value(
            row,
            "target_relevance_type",
            "not_available"
        )
    ).strip()


    # --------------------------------------------------------
    # Recommendation eligibility
    # --------------------------------------------------------

    recommendation_eligibility = (
        "evaluation_eligible"
        if feature_present
        else "audit_only"
    )


    # --------------------------------------------------------
    # Structured context
    # --------------------------------------------------------

    CONTEXT_ROWS.append({

        "dataset_id":
            dataset_id,

        "target":
            target,

        "feature":
            feature,

        "feature_type":
            feature_type,

        "missing_rate":
            safe_float(
                row["missing_rate"]
            ),

        "dataset_rows":
            dataset_rows,

        "dataset_columns":
            dataset_columns,

        "dataset_missing_rate":
            dataset_missing_rate,

        "feature_cardinality":
            feature_cardinality,

        "mean":
            mean_value,

        "variance":
            variance_value,

        "skewness":
            skewness_value,

        "outlier_rate":
            outlier_rate_value,

        "entropy":
            entropy_value,

        "target_relevance_type":
            target_relevance_type,

        "target_relevance":
            target_relevance_value,

        "mutual_information":
            mutual_information_value,

        "feature_present_in_training_representation":
            bool(feature_present),

        "recommendation_eligibility":
            recommendation_eligibility
    })


# ============================================================
# 4. CREATE DATAFRAMES
# ============================================================

CONTEXT_DF = pd.DataFrame(
    CONTEXT_ROWS
)


REPRESENTATION_WARNING_DF = pd.DataFrame(
    REPRESENTATION_WARNINGS,
    columns=[
        "dataset_id",
        "feature",
        "reason"
    ]
)


# ============================================================
# 5. STRUCTURAL VALIDATION
# ============================================================

if len(CONTEXT_DF) != len(
    RECOMMENDATION_PROFILE_DF
):

    raise RuntimeError(
        "Context population mismatch."
    )


if CONTEXT_DF.duplicated(
    subset=[
        "dataset_id",
        "feature"
    ]
).any():

    raise RuntimeError(
        "Duplicate dataset-feature contexts detected."
    )


# ============================================================
# 6. TARGET VALIDATION
# ============================================================

target_leakage = (
    CONTEXT_DF["feature"].str.lower()
    ==
    CONTEXT_DF["target"]
    .astype(str)
    .str.lower()
)


if target_leakage.any():

    raise RuntimeError(
        "Target leakage detected in LLM context."
    )


# ============================================================
# 7. TECHNICAL FEATURE VALIDATION
# ============================================================

technical_remaining = (
    set(
        CONTEXT_DF["feature"]
    )
    &
    TECHNICAL_FEATURES
)


if technical_remaining:

    raise RuntimeError(
        "Technical features remain in LLM context."
    )


# ============================================================
# 8. MISSINGNESS VALIDATION
# ============================================================

if not (
    CONTEXT_DF["missing_rate"] > 0
).all():

    raise RuntimeError(
        "Complete features entered the "
        "recommendation population."
    )


# ============================================================
# 9. FEATURE TYPE VALIDATION
# ============================================================

VALID_FEATURE_TYPES = {
    "categorical",
    "numerical"
}


invalid_feature_types = (
    set(
        CONTEXT_DF["feature_type"]
    )
    -
    VALID_FEATURE_TYPES
)


if invalid_feature_types:

    raise RuntimeError(
        "Invalid feature types detected."
    )


# ============================================================
# 10. SAVE CONTEXT
# ============================================================

CONTEXT_PATH = (
    CONTEXT_DIR
    /
    "llm_context.csv"
)


CONTEXT_DF.to_csv(
    CONTEXT_PATH,
    index=False
)


# ============================================================
# 11. SAVE REPRESENTATION AUDIT
# ============================================================

REPRESENTATION_WARNING_PATH = (
    CONTEXT_DIR
    /
    "representation_warnings.csv"
)


REPRESENTATION_WARNING_DF.to_csv(
    REPRESENTATION_WARNING_PATH,
    index=False
)


# ============================================================
# 12. SUMMARY COUNTS
# ============================================================

evaluation_eligible_count = int(
    (
        CONTEXT_DF[
            "recommendation_eligibility"
        ]
        ==
        "evaluation_eligible"
    ).sum()
)


audit_only_count = int(
    (
        CONTEXT_DF[
            "recommendation_eligibility"
        ]
        ==
        "audit_only"
    ).sum()
)


dataset_count = int(
    CONTEXT_DF[
        "dataset_id"
    ].nunique()
)


categorical_count = int(
    (
        CONTEXT_DF[
            "feature_type"
        ]
        ==
        "categorical"
    ).sum()
)


numerical_count = int(
    (
        CONTEXT_DF[
            "feature_type"
        ]
        ==
        "numerical"
    ).sum()
)


# ============================================================
# 13. FINAL SUMMARY
# ============================================================

print("\nCONTEXT GENERATION SUMMARY")
print("-" * 100)

print(
    f"Contexts generated      : "
    f"{len(CONTEXT_DF)}"
)

print(
    f"Datasets represented    : "
    f"{dataset_count}"
)

print(
    f"Categorical contexts    : "
    f"{categorical_count}"
)

print(
    f"Numerical contexts      : "
    f"{numerical_count}"
)

print(
    f"Evaluation-eligible     : "
    f"{evaluation_eligible_count}"
)

print(
    f"Audit-only              : "
    f"{audit_only_count}"
)

print(
    f"Representation warnings : "
    f"{len(REPRESENTATION_WARNING_DF)}"
)

print(
    "\nContext validation      : PASSED"
)


# ============================================================
# 14. CONTEXT TABLE
# ============================================================

print("\nCONTEXT TABLE")
print("-" * 100)

CONTEXT_TABLE_COLUMNS = [

    "dataset_id",
    "target",
    "feature",
    "feature_type",
    "missing_rate",
    "feature_cardinality",
    "target_relevance",
    "mutual_information",
    "feature_present_in_training_representation",
    "recommendation_eligibility"

]


display(
    CONTEXT_DF[
        CONTEXT_TABLE_COLUMNS
    ]
    .sort_values(
        [
            "dataset_id",
            "feature"
        ]
    )
    .reset_index(
        drop=True
    )
)


# ============================================================
# 15. REPRESENTATION WARNINGS
# ============================================================

if not REPRESENTATION_WARNING_DF.empty:

    print("\nREPRESENTATION WARNINGS")
    print("-" * 100)

    display(
        REPRESENTATION_WARNING_DF
    )

    print(
        "\nAudit-only features will not be evaluated "
        "by downstream recommendation/evaluation notebooks."
    )


# ============================================================
# 16. FINAL ASSERTIONS
# ============================================================

assert len(
    CONTEXT_DF
) == 12


assert (
    CONTEXT_DF[
        "feature"
    ]
    .isin(
        TECHNICAL_FEATURES
    )
    .sum()
    ==
    0
)


assert (
    CONTEXT_DF[
        "missing_rate"
    ]
    > 0
).all()


assert (
    CONTEXT_DF[
        "feature_present_in_training_representation"
    ]
    .sum()
    ==
    evaluation_eligible_count
)


assert evaluation_eligible_count == 11

assert audit_only_count == 1


# ============================================================
# 17. FINAL MESSAGE
# ============================================================

print("\n" + "=" * 100)
print("NOTEBOOK 05.4 CONTEXT GENERATION COMPLETE")
print("=" * 100)

print(
    f"LLM contexts ready  : "
    f"{len(CONTEXT_DF)}"
)

print(
    f"Evaluation eligible : "
    f"{evaluation_eligible_count}"
)

print(
    f"Audit-only          : "
    f"{audit_only_count}"
)

print(
    "Ready for Notebook 05.5 Eligibility Diagnostic."
)

print("=" * 100)

AIR-LLM — NOTEBOOK 05.4
BUILDING RESEARCH-GRADE LLM CONTEXT

CONTEXT GENERATION SUMMARY
----------------------------------------------------------------------------------------------------
Contexts generated      : 12
Datasets represented    : 2
Categorical contexts    : 12
Numerical contexts      : 0
Evaluation-eligible     : 11
Audit-only              : 1
Representation warnings : 1

Context validation      : PASSED

CONTEXT TABLE
----------------------------------------------------------------------------------------------------


,dataset_id,target,feature,feature_type,missing_rate,feature_cardinality,target_relevance,mutual_information,feature_present_in_training_representation,recommendation_eligibility
0,adult_income,income,native_country,categorical,0.017857,41,0.095968,0.007695,True,evaluation_eligible
1,adult_income,income,occupation,categorical,0.056204,14,0.349804,0.061990,True,evaluation_eligible
2,adult_income,income,workclass,categorical,0.055941,8,0.166965,0.016288,True,evaluation_eligible
3,diabetes_130us,readmitted,a1cresult,categorical,0.833230,3,0.011005,0.000397,False,audit_only
4,diabetes_130us,readmitted,diag_1,categorical,0.000211,684,0.129485,0.064322,True,evaluation_eligible
5,diabetes_130us,readmitted,diag_2,categorical,0.003425,691,0.109374,0.056227,True,evaluation_eligible
6,diabetes_130us,readmitted,diag_3,categorical,0.013883,745,0.104493,0.055781,True,evaluation_eligible
7,diabetes_130us,readmitted,max_glu_serum,categorical,0.947765,3,0.048610,0.000528,True,evaluation_eligible
8,diabetes_130us,readmitted,medical_specialty,categorical,0.490342,68,0.101634,0.011258,True,evaluation_eligible
9,diabetes_130us,readmitted,payer_code,categorical,0.395558,17,0.061686,0.005733,True,evaluation_eligible



REPRESENTATION WARNINGS
----------------------------------------------------------------------------------------------------


,dataset_id,feature,reason
0,diabetes_130us,a1cresult,Feature exists in Notebook 04 profile but is a...



Audit-only features will not be evaluated by downstream recommendation/evaluation notebooks.

NOTEBOOK 05.4 CONTEXT GENERATION COMPLETE
LLM contexts ready  : 12
Evaluation eligible : 11
Audit-only          : 1
Ready for Notebook 05.5 Eligibility Diagnostic.


In [6]:
# ============================================================
# NOTEBOOK 05.5 — ELIGIBILITY DIAGNOSTIC
# ============================================================

print("=" * 100)
print("AIR-LLM — NOTEBOOK 05.5")
print("ELIGIBILITY DIAGNOSTIC")
print("=" * 100)

diagnostic_columns = [
    "dataset_id",
    "target",
    "feature",
    "feature_type",
    "missing_rate",
    "feature_present_in_training_representation",
    "recommendation_eligibility"
]

display(
    CONTEXT_DF[
        diagnostic_columns
    ]
    .sort_values(
        [
            "dataset_id",
            "feature"
        ]
    )
    .reset_index(drop=True)
)

print("\nFEATURES CURRENTLY CLASSIFIED AS AUDIT-ONLY")
print("-" * 100)

display(
    CONTEXT_DF[
        CONTEXT_DF[
            "recommendation_eligibility"
        ]
        .astype(str)
        .str.strip()
        .str.lower()
        ==
        "audit_only"
    ][
        diagnostic_columns
    ]
    .reset_index(drop=True)
)

print("\nFEATURES NOT PRESENT IN TRAINING REPRESENTATION")
print("-" * 100)

display(
    CONTEXT_DF[
        ~CONTEXT_DF[
            "feature_present_in_training_representation"
        ]
        .astype(bool)
    ][
        diagnostic_columns
    ]
    .reset_index(drop=True)
)

AIR-LLM — NOTEBOOK 05.5
ELIGIBILITY DIAGNOSTIC


,dataset_id,target,feature,feature_type,missing_rate,feature_present_in_training_representation,recommendation_eligibility
0,adult_income,income,native_country,categorical,0.017857,True,evaluation_eligible
1,adult_income,income,occupation,categorical,0.056204,True,evaluation_eligible
2,adult_income,income,workclass,categorical,0.055941,True,evaluation_eligible
3,diabetes_130us,readmitted,a1cresult,categorical,0.833230,False,audit_only
4,diabetes_130us,readmitted,diag_1,categorical,0.000211,True,evaluation_eligible
5,diabetes_130us,readmitted,diag_2,categorical,0.003425,True,evaluation_eligible
6,diabetes_130us,readmitted,diag_3,categorical,0.013883,True,evaluation_eligible
7,diabetes_130us,readmitted,max_glu_serum,categorical,0.947765,True,evaluation_eligible
8,diabetes_130us,readmitted,medical_specialty,categorical,0.490342,True,evaluation_eligible
9,diabetes_130us,readmitted,payer_code,categorical,0.395558,True,evaluation_eligible



FEATURES CURRENTLY CLASSIFIED AS AUDIT-ONLY
----------------------------------------------------------------------------------------------------


,dataset_id,target,feature,feature_type,missing_rate,feature_present_in_training_representation,recommendation_eligibility
0,diabetes_130us,readmitted,a1cresult,categorical,0.83323,False,audit_only



FEATURES NOT PRESENT IN TRAINING REPRESENTATION
----------------------------------------------------------------------------------------------------


,dataset_id,target,feature,feature_type,missing_rate,feature_present_in_training_representation,recommendation_eligibility
0,diabetes_130us,readmitted,a1cresult,categorical,0.83323,False,audit_only


In [7]:
# ============================================================
# NOTEBOOK 05.5 — CONTEXT VALIDATION
# RESEARCH-GRADE FINAL VERSION
# ============================================================

print("=" * 100)
print("AIR-LLM — NOTEBOOK 05.5")
print("VALIDATING LLM CONTEXT")
print("=" * 100)


# ============================================================
# 1. REQUIRED CONTEXT COLUMNS
# ============================================================

REQUIRED_CONTEXT_COLUMNS = [
    "dataset_id",
    "target",
    "feature",
    "feature_type",
    "missing_rate",
    "dataset_rows",
    "dataset_columns",
    "dataset_missing_rate",
    "feature_cardinality",
    "target_relevance",
    "feature_present_in_training_representation",
    "recommendation_eligibility"
]

missing_columns = [
    column
    for column in REQUIRED_CONTEXT_COLUMNS
    if column not in CONTEXT_DF.columns
]

if missing_columns:
    raise RuntimeError(
        "Context is missing required columns:\n"
        + "\n".join(
            f"  - {column}"
            for column in missing_columns
        )
    )


# ============================================================
# 2. CONTEXT NON-EMPTY VALIDATION
# ============================================================

if CONTEXT_DF.empty:
    raise RuntimeError(
        "LLM context is empty."
    )


# ============================================================
# 3. TARGET LEAKAGE VALIDATION
# ============================================================

feature_values = (
    CONTEXT_DF["feature"]
    .astype(str)
    .str.strip()
    .str.lower()
)

target_values = (
    CONTEXT_DF["target"]
    .astype(str)
    .str.strip()
    .str.lower()
)

target_leakage_mask = (
    feature_values == target_values
)

if target_leakage_mask.any():

    leaked_targets = (
        CONTEXT_DF.loc[
            target_leakage_mask,
            [
                "dataset_id",
                "feature",
                "target"
            ]
        ]
        .drop_duplicates()
    )

    raise RuntimeError(
        "Target leakage detected:\n"
        + leaked_targets.to_string(
            index=False
        )
    )


# ============================================================
# 4. TECHNICAL FEATURE LEAKAGE VALIDATION
# ============================================================

technical_remaining = (
    set(
        CONTEXT_DF["feature"]
        .astype(str)
        .str.strip()
    )
    &
    set(TECHNICAL_FEATURES)
)

if technical_remaining:

    raise RuntimeError(
        "Technical feature leakage detected:\n"
        + "\n".join(
            f"  - {feature}"
            for feature in sorted(
                technical_remaining
            )
        )
    )


# ============================================================
# 5. MISSING-RATE VALIDATION
# ============================================================

CONTEXT_DF["missing_rate"] = pd.to_numeric(
    CONTEXT_DF["missing_rate"],
    errors="coerce"
)

if CONTEXT_DF["missing_rate"].isna().any():

    raise RuntimeError(
        "Invalid missing_rate values detected."
    )


if (
    (CONTEXT_DF["missing_rate"] < 0)
    |
    (CONTEXT_DF["missing_rate"] > 1)
).any():

    raise RuntimeError(
        "missing_rate values must be within [0, 1]."
    )


# ============================================================
# 6. ALL CONTEXT FEATURES MUST BE INCOMPLETE
# ============================================================

complete_features = CONTEXT_DF[
    CONTEXT_DF["missing_rate"] <= 0
]

if not complete_features.empty:

    raise RuntimeError(
        "Complete features entered the "
        "recommendation context:\n"
        +
        complete_features[
            [
                "dataset_id",
                "feature",
                "missing_rate"
            ]
        ].to_string(
            index=False
        )
    )


# ============================================================
# 7. FEATURE TYPE VALIDATION
# ============================================================

ALLOWED_FEATURE_TYPES = {
    "numerical",
    "categorical"
}

normalized_feature_types = (
    CONTEXT_DF["feature_type"]
    .astype(str)
    .str.strip()
    .str.lower()
)

invalid_types = (
    set(normalized_feature_types.unique())
    -
    ALLOWED_FEATURE_TYPES
)

if invalid_types:

    raise RuntimeError(
        "Invalid feature types detected:\n"
        + "\n".join(
            f"  - {feature_type}"
            for feature_type in sorted(
                invalid_types
            )
        )
    )


# ============================================================
# 8. TRAINING REPRESENTATION VALIDATION
# ============================================================

representation_series = (
    CONTEXT_DF[
        "feature_present_in_training_representation"
    ]
)

if representation_series.isna().any():

    raise RuntimeError(
        "Missing values detected in "
        "'feature_present_in_training_representation'."
    )

representation_values = set(
    representation_series
    .astype(bool)
    .unique()
)

if not representation_values.issubset(
    {True, False}
):

    raise RuntimeError(
        "Invalid values detected in "
        "'feature_present_in_training_representation'."
    )


# ============================================================
# 9. RECOMMENDATION ELIGIBILITY VALIDATION
# ============================================================

ALLOWED_ELIGIBILITY = {
    "evaluation_eligible",
    "audit_only"
}

eligibility_series = (
    CONTEXT_DF[
        "recommendation_eligibility"
    ]
    .astype(str)
    .str.strip()
    .str.lower()
)

invalid_eligibility = (
    set(eligibility_series.unique())
    -
    ALLOWED_ELIGIBILITY
)

if invalid_eligibility:

    raise RuntimeError(
        "Invalid recommendation eligibility values:\n"
        + "\n".join(
            f"  - {value}"
            for value in sorted(
                invalid_eligibility
            )
        )
    )


# ============================================================
# 10. REPRESENTATION / ELIGIBILITY CONSISTENCY
#
# Present in training representation
#     -> evaluation_eligible
#
# Absent from training representation
#     -> audit_only
# ============================================================

representation_eligibility_mismatch = CONTEXT_DF[
    (
        CONTEXT_DF[
            "feature_present_in_training_representation"
        ]
        &
        (
            eligibility_series
            !=
            "evaluation_eligible"
        )
    )
    |
    (
        ~CONTEXT_DF[
            "feature_present_in_training_representation"
        ]
        &
        (
            eligibility_series
            !=
            "audit_only"
        )
    )
]

if not representation_eligibility_mismatch.empty:

    raise RuntimeError(
        "Training representation and recommendation "
        "eligibility are inconsistent:\n"
        +
        representation_eligibility_mismatch[
            [
                "dataset_id",
                "feature",
                "feature_present_in_training_representation",
                "recommendation_eligibility"
            ]
        ].to_string(
            index=False
        )
    )


# ============================================================
# 11. TARGET REGISTRY CONSISTENCY
# ============================================================

for dataset_id, target in TARGET_REGISTRY.items():

    dataset_context = CONTEXT_DF[
        CONTEXT_DF["dataset_id"]
        == dataset_id
    ]

    if dataset_context.empty:
        continue

    invalid_target = (
        dataset_context["target"]
        .astype(str)
        .str.strip()
        !=
        str(target).strip()
    )

    if invalid_target.any():

        raise RuntimeError(
            f"Target registry mismatch detected "
            f"for dataset '{dataset_id}'."
        )


# ============================================================
# 12. DATASET DIMENSION VALIDATION
# ============================================================

for column in [
    "dataset_rows",
    "dataset_columns"
]:

    numeric_values = pd.to_numeric(
        CONTEXT_DF[column],
        errors="coerce"
    )

    if numeric_values.isna().any():

        raise RuntimeError(
            f"Invalid values detected in '{column}'."
        )

    if (numeric_values <= 0).any():

        raise RuntimeError(
            f"Non-positive values detected in '{column}'."
        )


# ============================================================
# 13. DATASET MISSING-RATE VALIDATION
# ============================================================

dataset_missing_rate = pd.to_numeric(
    CONTEXT_DF["dataset_missing_rate"],
    errors="coerce"
)

if dataset_missing_rate.isna().any():

    raise RuntimeError(
        "Invalid dataset_missing_rate values detected."
    )

if (
    (dataset_missing_rate < 0)
    |
    (dataset_missing_rate > 1)
).any():

    raise RuntimeError(
        "dataset_missing_rate values must be within [0, 1]."
    )


# ============================================================
# 14. FEATURE CARDINALITY VALIDATION
# ============================================================

feature_cardinality = pd.to_numeric(
    CONTEXT_DF["feature_cardinality"],
    errors="coerce"
)

if feature_cardinality.isna().any():

    raise RuntimeError(
        "Invalid feature_cardinality values detected."
    )

if (feature_cardinality <= 0).any():

    raise RuntimeError(
        "Feature cardinality must be positive."
    )


# ============================================================
# 15. DUPLICATE DATASET-FEATURE VALIDATION
# ============================================================

duplicate_mask = CONTEXT_DF.duplicated(
    subset=[
        "dataset_id",
        "feature"
    ],
    keep=False
)

if duplicate_mask.any():

    duplicates = (
        CONTEXT_DF.loc[
            duplicate_mask,
            [
                "dataset_id",
                "feature"
            ]
        ]
        .drop_duplicates()
    )

    raise RuntimeError(
        "Duplicate dataset-feature contexts detected:\n"
        +
        duplicates.to_string(
            index=False
        )
    )


# ============================================================
# 16. BUILD EVALUATION-ELIGIBLE CONTEXT
#
# Notebook 06 must use this population only.
# ============================================================

EVALUATION_CONTEXT_DF = (
    CONTEXT_DF[
        eligibility_series
        ==
        "evaluation_eligible"
    ]
    .copy()
    .reset_index(drop=True)
)


# ============================================================
# 17. BUILD AUDIT-ONLY CONTEXT
# ============================================================

AUDIT_ONLY_CONTEXT_DF = (
    CONTEXT_DF[
        eligibility_series
        ==
        "audit_only"
    ]
    .copy()
    .reset_index(drop=True)
)


# ============================================================
# 18. POPULATION VALIDATION
# ============================================================

EXPECTED_TOTAL_CONTEXTS = 12
EXPECTED_EVALUATION_CONTEXTS = 11
EXPECTED_AUDIT_ONLY_CONTEXTS = 1

observed_total = len(
    CONTEXT_DF
)

observed_evaluation = len(
    EVALUATION_CONTEXT_DF
)

observed_audit = len(
    AUDIT_ONLY_CONTEXT_DF
)


if observed_total != EXPECTED_TOTAL_CONTEXTS:

    raise RuntimeError(
        "Unexpected total context population: "
        f"expected {EXPECTED_TOTAL_CONTEXTS}, "
        f"observed {observed_total}."
    )


if observed_evaluation != EXPECTED_EVALUATION_CONTEXTS:

    raise RuntimeError(
        "Unexpected evaluation-eligible population: "
        f"expected {EXPECTED_EVALUATION_CONTEXTS}, "
        f"observed {observed_evaluation}."
    )


if observed_audit != EXPECTED_AUDIT_ONLY_CONTEXTS:

    raise RuntimeError(
        "Unexpected audit-only population: "
        f"expected {EXPECTED_AUDIT_ONLY_CONTEXTS}, "
        f"observed {observed_audit}."
    )


# ============================================================
# 19. EVALUATION CONTEXT MUST NEVER CONTAIN AUDIT-ONLY
# ============================================================

if (
    EVALUATION_CONTEXT_DF[
        "recommendation_eligibility"
    ]
    !=
    "evaluation_eligible"
).any():

    raise RuntimeError(
        "Audit-only features entered "
        "the evaluation population."
    )


# ============================================================
# 20. AUDIT CONTEXT MUST NEVER BE EVALUATED
# ============================================================

if (
    AUDIT_ONLY_CONTEXT_DF[
        "recommendation_eligibility"
    ]
    !=
    "audit_only"
).any():

    raise RuntimeError(
        "Non-audit features entered "
        "the audit-only population."
    )


# ============================================================
# 21. SAVE VALIDATED POPULATIONS
# ============================================================

EVALUATION_CONTEXT_PATH = (
    CONTEXT_DIR
    / "evaluation_context.csv"
)

AUDIT_ONLY_CONTEXT_PATH = (
    CONTEXT_DIR
    / "audit_only_context.csv"
)

EVALUATION_CONTEXT_DF.to_csv(
    EVALUATION_CONTEXT_PATH,
    index=False
)

AUDIT_ONLY_CONTEXT_DF.to_csv(
    AUDIT_ONLY_CONTEXT_PATH,
    index=False
)


# ============================================================
# 22. VALIDATION REPORT
# ============================================================

print("\nVALIDATION RESULTS")
print("-" * 100)

print(
    "Required context columns      : PASSED"
)

print(
    "Context non-empty             : PASSED"
)

print(
    "Target leakage                : PASSED"
)

print(
    "Technical leakage             : PASSED"
)

print(
    "Missing-rate validation       : PASSED"
)

print(
    "Incomplete-feature filter     : PASSED"
)

print(
    "Feature-type validation       : PASSED"
)

print(
    "Training representation       : PASSED"
)

print(
    "Recommendation eligibility    : PASSED"
)

print(
    "Representation consistency    : PASSED"
)

print(
    "Target registry consistency   : PASSED"
)

print(
    "Dataset dimensions            : PASSED"
)

print(
    "Dataset missing-rate check    : PASSED"
)

print(
    "Feature cardinality check     : PASSED"
)

print(
    "Duplicate context check       : PASSED"
)

print(
    "Evaluation population         : PASSED"
)

print(
    "Audit-only population         : PASSED"
)


# ============================================================
# 23. DATASET POPULATION REPORT
# ============================================================

print("\nDATASET POPULATION")
print("-" * 100)

print(
    f"{'Dataset':<20}"
    f"{'Total':>10}"
    f"{'Eligible':>12}"
    f"{'Audit-only':>12}"
)

print("-" * 100)

for dataset_id in TARGET_REGISTRY:

    dataset_contexts = CONTEXT_DF[
        CONTEXT_DF["dataset_id"]
        == dataset_id
    ]

    eligible_count = len(
        dataset_contexts[
            dataset_contexts[
                "recommendation_eligibility"
            ]
            ==
            "evaluation_eligible"
        ]
    )

    audit_count = len(
        dataset_contexts[
            dataset_contexts[
                "recommendation_eligibility"
            ]
            ==
            "audit_only"
        ]
    )

    print(
        f"{dataset_id:<20}"
        f"{len(dataset_contexts):>10}"
        f"{eligible_count:>12}"
        f"{audit_count:>12}"
    )


# ============================================================
# 24. AUDIT-ONLY FEATURES
# ============================================================

print("\nAUDIT-ONLY FEATURES")
print("-" * 100)

if AUDIT_ONLY_CONTEXT_DF.empty:

    print("None")

else:

    display(
        AUDIT_ONLY_CONTEXT_DF[
            [
                "dataset_id",
                "feature",
                "feature_present_in_training_representation",
                "recommendation_eligibility"
            ]
        ]
    )


# ============================================================
# 25. FINAL SUMMARY
# ============================================================

print("\n" + "=" * 100)
print("NOTEBOOK 05.5 CONTEXT VALIDATION COMPLETE")
print("=" * 100)

print(
    f"\nTotal contexts             : "
    f"{observed_total}"
)

print(
    f"Evaluation-eligible        : "
    f"{observed_evaluation}"
)

print(
    f"Audit-only                 : "
    f"{observed_audit}"
)

print(
    f"\nFull context file          : "
    f"{CONTEXT_PATH}"
)

print(
    f"Evaluation context file    : "
    f"{EVALUATION_CONTEXT_PATH}"
)

print(
    f"Audit-only context file    : "
    f"{AUDIT_ONLY_CONTEXT_PATH}"
)

print("\n" + "-" * 100)
print("NOTEBOOK 06 INPUT CONTRACT")
print("-" * 100)

print(
    "Notebook 06 must consume:"
)

print(
    f"  {EVALUATION_CONTEXT_PATH}"
)

print(
    "\nAudit-only contexts must NOT "
    "enter candidate evaluation."
)

print("\n" + "=" * 100)
print("STATUS: PASSED")
print("=" * 100)

AIR-LLM — NOTEBOOK 05.5
VALIDATING LLM CONTEXT

VALIDATION RESULTS
----------------------------------------------------------------------------------------------------
Required context columns      : PASSED
Context non-empty             : PASSED
Target leakage                : PASSED
Technical leakage             : PASSED
Missing-rate validation       : PASSED
Incomplete-feature filter     : PASSED
Feature-type validation       : PASSED
Training representation       : PASSED
Recommendation eligibility    : PASSED
Representation consistency    : PASSED
Target registry consistency   : PASSED
Dataset dimensions            : PASSED
Dataset missing-rate check    : PASSED
Feature cardinality check     : PASSED
Duplicate context check       : PASSED
Evaluation population         : PASSED
Audit-only population         : PASSED

DATASET POPULATION
----------------------------------------------------------------------------------------------------
Dataset                  Total    Eligible  Audi

,dataset_id,feature,feature_present_in_training_representation,recommendation_eligibility
0,diabetes_130us,a1cresult,False,audit_only



NOTEBOOK 05.5 CONTEXT VALIDATION COMPLETE

Total contexts             : 12
Evaluation-eligible        : 11
Audit-only                 : 1

Full context file          : /content/drive/MyDrive/AIR_LLM_Research/data/notebook_05/contexts/llm_context.csv
Evaluation context file    : /content/drive/MyDrive/AIR_LLM_Research/data/notebook_05/contexts/evaluation_context.csv
Audit-only context file    : /content/drive/MyDrive/AIR_LLM_Research/data/notebook_05/contexts/audit_only_context.csv

----------------------------------------------------------------------------------------------------
NOTEBOOK 06 INPUT CONTRACT
----------------------------------------------------------------------------------------------------
Notebook 06 must consume:
  /content/drive/MyDrive/AIR_LLM_Research/data/notebook_05/contexts/evaluation_context.csv

Audit-only contexts must NOT enter candidate evaluation.

STATUS: PASSED


In [8]:
# ============================================================
# NOTEBOOK 05.6 — CANDIDATE STRATEGY POOL
# AIR-LLM RESEARCH-GRADE FINAL VERSION
# ============================================================

print("=" * 100)
print("AIR-LLM — NOTEBOOK 05.6")
print("DEFINING RESEARCH CANDIDATE STRATEGY POOL")
print("=" * 100)

# ============================================================
# 1. PURPOSE AND RESEARCH CONTRACT
# ============================================================

# This notebook defines the complete candidate strategy registry
# used by AIR-LLM for downstream missing-value imputation.
#
# This notebook DOES NOT:
#   - perform imputation
#   - train models
#   - evaluate imputation quality
#   - select a final strategy
#   - use the target variable for imputation
#
# Notebook 05.7:
#   Constructs feature-compatible candidate sets.
#
# Notebook 06:
#   Executes candidate strategies and performs candidate evaluation.
#
# Notebook 07:
#   Performs adaptive utility-based final strategy selection.
#
# Candidate strategies are deliberately divided into:
#   1. Univariate statistical baselines
#   2. Distribution-preserving stochastic baseline
#   3. Multivariate predictive / iterative ML methods
#   4. Low-rank matrix-completion methods
#
# The target variable is NEVER used as an imputation predictor.
#
# Temporal propagation methods are intentionally excluded because
# the AIR-LLM experimental datasets are non-temporal tabular data.

RANDOM_STATE = 42

FEATURE_TYPE_NUMERICAL = "numerical"
FEATURE_TYPE_CATEGORICAL = "categorical"

ALLOWED_FEATURE_TYPES = {
    FEATURE_TYPE_NUMERICAL,
    FEATURE_TYPE_CATEGORICAL
}

ALLOWED_STRATEGY_FAMILIES = {
    "statistical",
    "machine_learning"
}

ALLOWED_IMPLEMENTATION_STATUS = {
    "planned",
    "implemented",
    "validated",
    "disabled"
}


# ============================================================
# 2. CANDIDATE STRATEGY REGISTRY
# ============================================================

CANDIDATE_STRATEGIES = [

    # ========================================================
    # A. UNIVARIATE STATISTICAL BASELINES
    # ========================================================

    {
        "strategy_id": "mean",
        "family": "statistical",
        "subfamily": "univariate_location",
        "feature_types": [FEATURE_TYPE_NUMERICAL],
        "requires_multivariate_data": False,
        "stochastic": False,
        "requires_target": False,
        "research_baseline": True,
        "implementation_status": "planned",
        "description":
            "Mean imputation using the arithmetic mean of observed "
            "values of the incomplete numerical feature."
    },

    {
        "strategy_id": "median",
        "family": "statistical",
        "subfamily": "univariate_location",
        "feature_types": [FEATURE_TYPE_NUMERICAL],
        "requires_multivariate_data": False,
        "stochastic": False,
        "requires_target": False,
        "research_baseline": True,
        "implementation_status": "planned",
        "description":
            "Median imputation using the observed median of the "
            "incomplete numerical feature."
    },

    {
        "strategy_id": "mode",
        "family": "statistical",
        "subfamily": "univariate_frequency",
        "feature_types": [FEATURE_TYPE_CATEGORICAL],
        "requires_multivariate_data": False,
        "stochastic": False,
        "requires_target": False,
        "research_baseline": True,
        "implementation_status": "planned",
        "description":
            "Mode imputation using the most frequent observed "
            "category of the incomplete categorical feature."
    },

    {
        "strategy_id": "constant",
        "family": "statistical",
        "subfamily": "fixed_value",
        "feature_types": [
            FEATURE_TYPE_NUMERICAL,
            FEATURE_TYPE_CATEGORICAL
        ],
        "requires_multivariate_data": False,
        "stochastic": False,
        "requires_target": False,
        "research_baseline": True,
        "implementation_status": "planned",
        "description":
            "Deterministic constant-value imputation using a "
            "predefined feature-type-specific replacement policy."
    },

    {
        "strategy_id": "random_sample",
        "family": "statistical",
        "subfamily": "distribution_preserving",
        "feature_types": [
            FEATURE_TYPE_NUMERICAL,
            FEATURE_TYPE_CATEGORICAL
        ],
        "requires_multivariate_data": False,
        "stochastic": True,
        "requires_target": False,
        "research_baseline": True,
        "implementation_status": "planned",
        "description":
            "Stochastic marginal imputation by sampling from the "
            "observed distribution of the incomplete feature."
    },


    # ========================================================
    # B. MULTIVARIATE MACHINE-LEARNING METHODS
    # ========================================================

    {
        "strategy_id": "knn",
        "family": "machine_learning",
        "subfamily": "nearest_neighbor",
        "feature_types": [FEATURE_TYPE_NUMERICAL],
        "requires_multivariate_data": True,
        "stochastic": False,
        "requires_target": False,
        "research_baseline": True,
        "implementation_status": "planned",
        "description":
            "K-nearest-neighbor imputation using multivariate "
            "similarity among observed numerical features."
    },

    {
        "strategy_id": "iterative",
        "family": "machine_learning",
        "subfamily": "iterative_regression",
        "feature_types": [FEATURE_TYPE_NUMERICAL],
        "requires_multivariate_data": True,
        "stochastic": False,
        "requires_target": False,
        "research_baseline": True,
        "implementation_status": "planned",
        "description":
            "Iterative regression-based imputation using observed "
            "relationships among numerical predictors."
    },

    {
        "strategy_id": "mice",
        "family": "machine_learning",
        "subfamily": "multiple_imputation",
        "feature_types": [
            FEATURE_TYPE_NUMERICAL,
            FEATURE_TYPE_CATEGORICAL
        ],
        "requires_multivariate_data": True,
        "stochastic": True,
        "requires_target": False,
        "research_baseline": True,
        "implementation_status": "planned",
        "description":
            "Multiple imputation by chained equations using "
            "iterative conditional models appropriate to variable type."
    },

    {
        "strategy_id": "random_forest",
        "family": "machine_learning",
        "subfamily": "tree_ensemble",
        "feature_types": [
            FEATURE_TYPE_NUMERICAL,
            FEATURE_TYPE_CATEGORICAL
        ],
        "requires_multivariate_data": True,
        "stochastic": True,
        "requires_target": False,
        "research_baseline": True,
        "implementation_status": "planned",
        "description":
            "Random-forest predictive imputation using nonlinear "
            "multivariate relationships after controlled categorical encoding."
    },

    {
        "strategy_id": "gradient_boosting",
        "family": "machine_learning",
        "subfamily": "boosted_tree",
        "feature_types": [
            FEATURE_TYPE_NUMERICAL,
            FEATURE_TYPE_CATEGORICAL
        ],
        "requires_multivariate_data": True,
        "stochastic": False,
        "requires_target": False,
        "research_baseline": True,
        "implementation_status": "planned",
        "description":
            "Gradient-boosted predictive imputation using nonlinear "
            "multivariate relationships after controlled categorical encoding."
    },

    {
        "strategy_id": "missforest",
        "family": "machine_learning",
        "subfamily": "iterative_tree_ensemble",
        "feature_types": [
            FEATURE_TYPE_NUMERICAL,
            FEATURE_TYPE_CATEGORICAL
        ],
        "requires_multivariate_data": True,
        "stochastic": True,
        "requires_target": False,
        "research_baseline": True,
        "implementation_status": "planned",
        "description":
            "Iterative random-forest imputation for mixed-type tabular "
            "data using repeated predictive reconstruction."
    },


    # ========================================================
    # C. LOW-RANK / MATRIX COMPLETION
    # ========================================================

    {
        "strategy_id": "softimpute",
        "family": "machine_learning",
        "subfamily": "low_rank_matrix_completion",
        "feature_types": [FEATURE_TYPE_NUMERICAL],
        "requires_multivariate_data": True,
        "stochastic": False,
        "requires_target": False,
        "research_baseline": False,
        "implementation_status": "planned",
        "description":
            "Low-rank matrix completion using soft-thresholded "
            "singular-value decomposition for numerical features."
    },

    {
        "strategy_id": "matrix_factorization",
        "family": "machine_learning",
        "subfamily": "latent_factor_model",
        "feature_types": [FEATURE_TYPE_NUMERICAL],
        "requires_multivariate_data": True,
        "stochastic": False,
        "requires_target": False,
        "research_baseline": False,
        "implementation_status": "planned",
        "description":
            "Latent-factor matrix completion for reconstructing "
            "missing numerical observations."
    }
]


# ============================================================
# 3. BUILD REGISTRY DATAFRAME
# ============================================================

CANDIDATE_STRATEGY_DF = pd.DataFrame(
    CANDIDATE_STRATEGIES
)

CANDIDATE_STRATEGY_DF["random_state"] = RANDOM_STATE


# ============================================================
# 4. REQUIRED REGISTRY COLUMNS
# ============================================================

REQUIRED_STRATEGY_COLUMNS = [

    "strategy_id",
    "family",
    "subfamily",
    "feature_types",
    "requires_multivariate_data",
    "stochastic",
    "requires_target",
    "research_baseline",
    "implementation_status",
    "description",
    "random_state"
]

missing_strategy_columns = [
    column
    for column in REQUIRED_STRATEGY_COLUMNS
    if column not in CANDIDATE_STRATEGY_DF.columns
]

if missing_strategy_columns:

    raise RuntimeError(
        "Candidate strategy registry is missing required columns:\n"
        + "\n".join(
            f"  - {column}"
            for column in missing_strategy_columns
        )
    )


# ============================================================
# 5. STRATEGY ID VALIDATION
# ============================================================

if CANDIDATE_STRATEGY_DF["strategy_id"].isna().any():

    raise RuntimeError(
        "Missing strategy IDs detected."
    )

if (
    CANDIDATE_STRATEGY_DF["strategy_id"]
    .astype(str)
    .str.strip()
    .eq("")
    .any()
):

    raise RuntimeError(
        "Empty strategy IDs detected."
    )

if CANDIDATE_STRATEGY_DF["strategy_id"].duplicated().any():

    duplicate_ids = (
        CANDIDATE_STRATEGY_DF.loc[
            CANDIDATE_STRATEGY_DF["strategy_id"].duplicated(
                keep=False
            ),
            "strategy_id"
        ]
        .astype(str)
        .unique()
        .tolist()
    )

    raise RuntimeError(
        "Duplicate strategy IDs detected:\n"
        + "\n".join(
            f"  - {value}"
            for value in duplicate_ids
        )
    )


# ============================================================
# 6. FAMILY VALIDATION
# ============================================================

invalid_families = (
    set(
        CANDIDATE_STRATEGY_DF["family"]
        .astype(str)
        .str.strip()
        .str.lower()
        .unique()
    )
    -
    ALLOWED_STRATEGY_FAMILIES
)

if invalid_families:

    raise RuntimeError(
        "Invalid strategy families detected:\n"
        + "\n".join(
            f"  - {value}"
            for value in sorted(invalid_families)
        )
    )


# ============================================================
# 7. FEATURE-TYPE VALIDATION
# ============================================================

for _, strategy in CANDIDATE_STRATEGY_DF.iterrows():

    strategy_id = strategy["strategy_id"]
    feature_types = strategy["feature_types"]

    if not isinstance(feature_types, list):

        raise RuntimeError(
            f"Strategy '{strategy_id}' feature_types "
            "must be a list."
        )

    if len(feature_types) == 0:

        raise RuntimeError(
            f"Strategy '{strategy_id}' has no supported feature types."
        )

    invalid_types = (
        set(feature_types)
        -
        ALLOWED_FEATURE_TYPES
    )

    if invalid_types:

        raise RuntimeError(
            f"Strategy '{strategy_id}' contains invalid "
            f"feature types: {invalid_types}"
        )

    if len(feature_types) != len(set(feature_types)):

        raise RuntimeError(
            f"Strategy '{strategy_id}' contains duplicate "
            "feature types."
        )


# ============================================================
# 8. BOOLEAN FIELD VALIDATION
# ============================================================

BOOLEAN_COLUMNS = [

    "requires_multivariate_data",
    "stochastic",
    "requires_target",
    "research_baseline"
]

for column in BOOLEAN_COLUMNS:

    if CANDIDATE_STRATEGY_DF[column].isna().any():

        raise RuntimeError(
            f"Missing boolean values detected in '{column}'."
        )

    invalid_boolean = (
        ~CANDIDATE_STRATEGY_DF[column]
        .map(lambda value: isinstance(value, (bool, np.bool_)))
    )

    if invalid_boolean.any():

        raise RuntimeError(
            f"Invalid boolean values detected in '{column}'."
        )


# ============================================================
# 9. IMPLEMENTATION STATUS VALIDATION
# ============================================================

invalid_statuses = (
    set(
        CANDIDATE_STRATEGY_DF["implementation_status"]
        .astype(str)
        .str.strip()
        .str.lower()
        .unique()
    )
    -
    ALLOWED_IMPLEMENTATION_STATUS
)

if invalid_statuses:

    raise RuntimeError(
        "Invalid implementation statuses detected:\n"
        + "\n".join(
            f"  - {value}"
            for value in sorted(invalid_statuses)
        )
    )


# ============================================================
# 10. DESCRIPTION VALIDATION
# ============================================================

if CANDIDATE_STRATEGY_DF["description"].isna().any():

    raise RuntimeError(
        "Missing strategy descriptions detected."
    )

if (
    CANDIDATE_STRATEGY_DF["description"]
    .astype(str)
    .str.strip()
    .eq("")
    .any()
):

    raise RuntimeError(
        "Empty strategy descriptions detected."
    )


# ============================================================
# 11. TARGET-LEAKAGE VALIDATION
# ============================================================

# Candidate imputation must not use the target variable.
# The target may be used later for evaluation, but not for
# reconstructing missing feature values.

if (
    CANDIDATE_STRATEGY_DF["requires_target"]
    .fillna(False)
    .any()
):

    target_dependent = (
        CANDIDATE_STRATEGY_DF.loc[
            CANDIDATE_STRATEGY_DF["requires_target"],
            "strategy_id"
        ]
        .tolist()
    )

    raise RuntimeError(
        "Target-dependent imputation strategies detected:\n"
        + "\n".join(
            f"  - {strategy}"
            for strategy in target_dependent
        )
    )


# ============================================================
# 12. MULTIVARIATE CONSISTENCY VALIDATION
# ============================================================

# Any strategy requiring multivariate information must support
# at least one feature type and must not be classified as a
# purely univariate strategy.

UNIVARIATE_SUBFAMILIES = {
    "univariate_location",
    "univariate_frequency",
    "fixed_value",
    "distribution_preserving"
}

for _, strategy in CANDIDATE_STRATEGY_DF.iterrows():

    if (
        strategy["subfamily"] not in UNIVARIATE_SUBFAMILIES
        and
        strategy["requires_multivariate_data"] is False
    ):

        # Only explicitly defined univariate strategies may be
        # marked as non-multivariate.
        raise RuntimeError(
            f"Strategy '{strategy['strategy_id']}' has an "
            "inconsistent multivariate-data declaration."
        )


# ============================================================
# 13. LOW-RANK METHOD CONSISTENCY
# ============================================================

LOW_RANK_STRATEGIES = {
    "softimpute",
    "matrix_factorization"
}

for strategy_id in LOW_RANK_STRATEGIES:

    row = CANDIDATE_STRATEGY_DF[
        CANDIDATE_STRATEGY_DF["strategy_id"]
        == strategy_id
    ]

    if row.empty:
        raise RuntimeError(
            f"Required low-rank strategy '{strategy_id}' "
            "is missing."
        )

    feature_types = row.iloc[0]["feature_types"]

    if feature_types != [FEATURE_TYPE_NUMERICAL]:

        raise RuntimeError(
            f"Low-rank strategy '{strategy_id}' must be "
            "numerical-only."
        )


# ============================================================
# 14. REQUIRED STRATEGY COVERAGE
# ============================================================

REQUIRED_STRATEGY_IDS = {

    "mean",
    "median",
    "mode",
    "constant",
    "random_sample",
    "knn",
    "iterative",
    "mice",
    "random_forest",
    "gradient_boosting",
    "missforest",
    "softimpute",
    "matrix_factorization"
}

observed_strategy_ids = set(
    CANDIDATE_STRATEGY_DF["strategy_id"]
)

missing_required_strategies = (
    REQUIRED_STRATEGY_IDS
    -
    observed_strategy_ids
)

if missing_required_strategies:

    raise RuntimeError(
        "Required candidate strategies are missing:\n"
        + "\n".join(
            f"  - {strategy}"
            for strategy in sorted(
                missing_required_strategies
            )
        )
    )


# ============================================================
# 15. STRATEGY POPULATION VALIDATION
# ============================================================

EXPECTED_STRATEGY_COUNT = 13

if len(CANDIDATE_STRATEGY_DF) != EXPECTED_STRATEGY_COUNT:

    raise RuntimeError(
        "Unexpected candidate strategy population: "
        f"expected {EXPECTED_STRATEGY_COUNT}, "
        f"observed {len(CANDIDATE_STRATEGY_DF)}."
    )


# ============================================================
# 16. RANDOM-STATE VALIDATION
# ============================================================

if not isinstance(RANDOM_STATE, int):

    raise RuntimeError(
        "RANDOM_STATE must be an integer."
    )

if RANDOM_STATE < 0:

    raise RuntimeError(
        "RANDOM_STATE must be non-negative."
    )

if (
    CANDIDATE_STRATEGY_DF["random_state"]
    .nunique()
    != 1
):

    raise RuntimeError(
        "Candidate strategies do not share a common random state."
    )


# ============================================================
# 17. FEATURE-TYPE COVERAGE
# ============================================================

FEATURE_TYPE_COVERAGE = []

for feature_type in [
    FEATURE_TYPE_NUMERICAL,
    FEATURE_TYPE_CATEGORICAL
]:

    compatible = CANDIDATE_STRATEGY_DF[
        CANDIDATE_STRATEGY_DF["feature_types"].apply(
            lambda values:
                feature_type in values
        )
    ]

    FEATURE_TYPE_COVERAGE.append({

        "feature_type": feature_type,

        "candidate_count":
            len(compatible),

        "candidate_strategies":
            ", ".join(
                compatible["strategy_id"]
                .tolist()
            )
    })


FEATURE_TYPE_COVERAGE_DF = pd.DataFrame(
    FEATURE_TYPE_COVERAGE
)


# ============================================================
# 18. COVERAGE VALIDATION
# ============================================================

for feature_type in [
    FEATURE_TYPE_NUMERICAL,
    FEATURE_TYPE_CATEGORICAL
]:

    count = int(
        FEATURE_TYPE_COVERAGE_DF.loc[
            FEATURE_TYPE_COVERAGE_DF["feature_type"]
            == feature_type,
            "candidate_count"
        ].iloc[0]
    )

    if count == 0:

        raise RuntimeError(
            f"No candidate strategies support "
            f"feature type '{feature_type}'."
        )


# ============================================================
# 19. STRATEGY FAMILY COVERAGE
# ============================================================

STRATEGY_FAMILY_DF = (
    CANDIDATE_STRATEGY_DF
    .groupby(
        [
            "family",
            "research_baseline"
        ],
        as_index=False
    )
    .size()
    .rename(
        columns={
            "size": "strategy_count"
        }
    )
)


# ============================================================
# 20. STRATEGY COUNTS
# ============================================================

total_strategies = len(
    CANDIDATE_STRATEGY_DF
)

statistical_count = int(
    (
        CANDIDATE_STRATEGY_DF["family"]
        == "statistical"
    ).sum()
)

machine_learning_count = int(
    (
        CANDIDATE_STRATEGY_DF["family"]
        == "machine_learning"
    ).sum()
)

numerical_capable_count = int(
    CANDIDATE_STRATEGY_DF["feature_types"].apply(
        lambda values:
            FEATURE_TYPE_NUMERICAL in values
    ).sum()
)

categorical_capable_count = int(
    CANDIDATE_STRATEGY_DF["feature_types"].apply(
        lambda values:
            FEATURE_TYPE_CATEGORICAL in values
    ).sum()
)

multivariate_count = int(
    CANDIDATE_STRATEGY_DF[
        "requires_multivariate_data"
    ].sum()
)

research_baseline_count = int(
    CANDIDATE_STRATEGY_DF[
        "research_baseline"
    ].sum()
)

deterministic_count = int(
    (
        ~CANDIDATE_STRATEGY_DF["stochastic"]
    ).sum()
)

stochastic_count = int(
    CANDIDATE_STRATEGY_DF[
        "stochastic"
    ].sum()
)


# ============================================================
# 21. SAVE STRATEGY REGISTRY
# ============================================================

STRATEGY_POOL_PATH = (
    CONTEXT_DIR
    / "candidate_strategy_pool.csv"
)

CANDIDATE_STRATEGY_DF.to_csv(
    STRATEGY_POOL_PATH,
    index=False
)


# ============================================================
# 22. VALIDATION SUMMARY
# ============================================================

print("\nCANDIDATE STRATEGY POOL SUMMARY")
print("-" * 100)

print(
    f"Total strategies              : "
    f"{total_strategies}"
)

print(
    f"Statistical strategies        : "
    f"{statistical_count}"
)

print(
    f"Machine-learning strategies   : "
    f"{machine_learning_count}"
)

print(
    f"Numerical-capable strategies  : "
    f"{numerical_capable_count}"
)

print(
    f"Categorical-capable strategies: "
    f"{categorical_capable_count}"
)

print(
    f"Multivariate strategies       : "
    f"{multivariate_count}"
)

print(
    f"Research baselines            : "
    f"{research_baseline_count}"
)

print(
    f"Deterministic strategies      : "
    f"{deterministic_count}"
)

print(
    f"Stochastic strategies         : "
    f"{stochastic_count}"
)

print(
    f"Random state                  : "
    f"{RANDOM_STATE}"
)


# ============================================================
# 23. VALIDATION REPORT
# ============================================================

print("\nREGISTRY VALIDATION")
print("-" * 100)

print(
    "Required columns             : PASSED"
)

print(
    "Strategy ID validation       : PASSED"
)

print(
    "Strategy family validation   : PASSED"
)

print(
    "Feature-type validation      : PASSED"
)

print(
    "Boolean field validation     : PASSED"
)

print(
    "Implementation-status check  : PASSED"
)

print(
    "Target-leakage check         : PASSED"
)

print(
    "Multivariate consistency     : PASSED"
)

print(
    "Low-rank consistency         : PASSED"
)

print(
    "Random-state validation      : PASSED"
)

print(
    "Strategy population check    : PASSED"
)

print(
    "Required strategy coverage   : PASSED"
)

print(
    "Description validation       : PASSED"
)

print(
    "Feature-type coverage        : PASSED"
)


# ============================================================
# 24. STRATEGY POOL
# ============================================================

print("\nSTRATEGY POOL")
print("-" * 100)

display(
    CANDIDATE_STRATEGY_DF[
        [
            "strategy_id",
            "family",
            "subfamily",
            "feature_types",
            "requires_multivariate_data",
            "stochastic",
            "requires_target",
            "research_baseline",
            "implementation_status"
        ]
    ]
)


# ============================================================
# 25. FEATURE-TYPE COVERAGE
# ============================================================

print("\nFEATURE-TYPE COVERAGE")
print("-" * 100)

display(
    FEATURE_TYPE_COVERAGE_DF
)


# ============================================================
# 26. STRATEGY FAMILY COVERAGE
# ============================================================

print("\nSTRATEGY FAMILY COVERAGE")
print("-" * 100)

display(
    STRATEGY_FAMILY_DF
)


# ============================================================
# 27. RESEARCH STRATEGY DEFINITIONS
# ============================================================

print("\nRESEARCH STRATEGY DEFINITIONS")
print("-" * 100)

display(
    CANDIDATE_STRATEGY_DF[
        [
            "strategy_id",
            "family",
            "subfamily",
            "description"
        ]
    ]
    .sort_values("strategy_id")
    .reset_index(drop=True)
)


# ============================================================
# 28. FINAL STATUS
# ============================================================

print("\n" + "=" * 100)
print("NOTEBOOK 05.6 CANDIDATE STRATEGY POOL COMPLETE")
print("=" * 100)

print(
    f"Candidate strategies ready : "
    f"{total_strategies}"
)

print(
    f"Strategy registry           : "
    f"{STRATEGY_POOL_PATH}"
)

print(
    "\nFeature-type standardization:"
)

print(
    "  Numerical   -> 'numerical'"
)

print(
    "  Categorical -> 'categorical'"
)

print(
    "\nNotebook 05.7 may now construct "
    "feature-compatible candidate sets."
)

print(
    "Notebook 06 will execute and evaluate "
    "the resulting candidate strategies."
)

print("=" * 100)

AIR-LLM — NOTEBOOK 05.6
DEFINING RESEARCH CANDIDATE STRATEGY POOL

CANDIDATE STRATEGY POOL SUMMARY
----------------------------------------------------------------------------------------------------
Total strategies              : 13
Statistical strategies        : 5
Machine-learning strategies   : 8
Numerical-capable strategies  : 12
Categorical-capable strategies: 7
Multivariate strategies       : 8
Research baselines            : 11
Deterministic strategies      : 9
Stochastic strategies         : 4
Random state                  : 42

REGISTRY VALIDATION
----------------------------------------------------------------------------------------------------
Required columns             : PASSED
Strategy ID validation       : PASSED
Strategy family validation   : PASSED
Feature-type validation      : PASSED
Boolean field validation     : PASSED
Implementation-status check  : PASSED
Target-leakage check         : PASSED
Multivariate consistency     : PASSED
Low-rank consistency         :

,strategy_id,family,subfamily,feature_types,requires_multivariate_data,stochastic,requires_target,research_baseline,implementation_status
0,mean,statistical,univariate_location,[numerical],False,False,False,True,planned
1,median,statistical,univariate_location,[numerical],False,False,False,True,planned
2,mode,statistical,univariate_frequency,[categorical],False,False,False,True,planned
3,constant,statistical,fixed_value,"[numerical, categorical]",False,False,False,True,planned
4,random_sample,statistical,distribution_preserving,"[numerical, categorical]",False,True,False,True,planned
5,knn,machine_learning,nearest_neighbor,[numerical],True,False,False,True,planned
6,iterative,machine_learning,iterative_regression,[numerical],True,False,False,True,planned
7,mice,machine_learning,multiple_imputation,"[numerical, categorical]",True,True,False,True,planned
8,random_forest,machine_learning,tree_ensemble,"[numerical, categorical]",True,True,False,True,planned
9,gradient_boosting,machine_learning,boosted_tree,"[numerical, categorical]",True,False,False,True,planned



FEATURE-TYPE COVERAGE
----------------------------------------------------------------------------------------------------


,feature_type,candidate_count,candidate_strategies
0,numerical,12,"mean, median, constant, random_sample, knn, it..."
1,categorical,7,"mode, constant, random_sample, mice, random_fo..."



STRATEGY FAMILY COVERAGE
----------------------------------------------------------------------------------------------------


,family,research_baseline,strategy_count
0,machine_learning,False,2
1,machine_learning,True,6
2,statistical,True,5



RESEARCH STRATEGY DEFINITIONS
----------------------------------------------------------------------------------------------------


,strategy_id,family,subfamily,description
0,constant,statistical,fixed_value,Deterministic constant-value imputation using ...
1,gradient_boosting,machine_learning,boosted_tree,Gradient-boosted predictive imputation using n...
2,iterative,machine_learning,iterative_regression,Iterative regression-based imputation using ob...
3,knn,machine_learning,nearest_neighbor,K-nearest-neighbor imputation using multivaria...
4,matrix_factorization,machine_learning,latent_factor_model,Latent-factor matrix completion for reconstruc...
5,mean,statistical,univariate_location,Mean imputation using the arithmetic mean of o...
6,median,statistical,univariate_location,Median imputation using the observed median of...
7,mice,machine_learning,multiple_imputation,Multiple imputation by chained equations using...
8,missforest,machine_learning,iterative_tree_ensemble,Iterative random-forest imputation for mixed-t...
9,mode,statistical,univariate_frequency,Mode imputation using the most frequent observ...



NOTEBOOK 05.6 CANDIDATE STRATEGY POOL COMPLETE
Candidate strategies ready : 13
Strategy registry           : /content/drive/MyDrive/AIR_LLM_Research/data/notebook_05/contexts/candidate_strategy_pool.csv

Feature-type standardization:
  Numerical   -> 'numerical'
  Categorical -> 'categorical'

Notebook 05.7 may now construct feature-compatible candidate sets.
Notebook 06 will execute and evaluate the resulting candidate strategies.


In [9]:
# ============================================================
# NOTEBOOK 05.7 — CANDIDATE POOL VALIDATION
# ============================================================

print("=" * 100)
print("AIR-LLM — NOTEBOOK 05.7")
print("VALIDATING CANDIDATE STRATEGY POOL")
print("=" * 100)


# ============================================================
# 1. REQUIRED DEPENDENCIES
# ============================================================

import os
import ast
import json
import numpy as np
import pandas as pd


# ============================================================
# 2. REQUIRED STRATEGY COLUMNS
# ============================================================

required_strategy_columns = [
    "strategy_id",
    "family",
    "subfamily",
    "feature_types",
    "requires_multivariate_data",
    "stochastic",
    "requires_target",
    "research_baseline",
    "implementation_status",
    "description",
    "random_state"
]


if "CANDIDATE_STRATEGY_DF" not in globals():

    raise RuntimeError(
        "CANDIDATE_STRATEGY_DF is not available. "
        "Execute Notebook 05.6 before Notebook 05.7."
    )


missing_strategy_columns = [
    column
    for column in required_strategy_columns
    if column not in CANDIDATE_STRATEGY_DF.columns
]


if missing_strategy_columns:

    raise RuntimeError(
        "Candidate strategy pool is malformed. "
        "Missing required columns:\n"
        + "\n".join(missing_strategy_columns)
    )


# ============================================================
# 3. COPY REGISTRY
# ============================================================

VALIDATED_CANDIDATE_STRATEGY_DF = (
    CANDIDATE_STRATEGY_DF.copy()
)


# ============================================================
# 4. NON-EMPTY POOL
# ============================================================

if VALIDATED_CANDIDATE_STRATEGY_DF.empty:

    raise RuntimeError(
        "Candidate strategy pool is empty."
    )


# ============================================================
# 5. FEATURE-TYPE STANDARDIZATION
# ============================================================

STANDARD_FEATURE_TYPES = {
    "numerical",
    "categorical"
}


def normalize_feature_types(value):

    if isinstance(value, (list, tuple, set)):

        values = list(value)

    elif isinstance(value, str):

        value = value.strip()

        try:
            parsed = ast.literal_eval(value)

            if isinstance(
                parsed,
                (list, tuple, set)
            ):

                values = list(parsed)

            else:

                values = [
                    value
                ]

        except Exception:

            values = [
                value
            ]

    else:

        raise ValueError(
            "Unsupported feature_types representation."
        )

    normalized = []

    for feature_type in values:

        feature_type = str(
            feature_type
        ).strip().lower()

        # Standardize legacy terminology
        if feature_type == "numeric":

            feature_type = "numerical"

        normalized.append(
            feature_type
        )

    return sorted(
        set(normalized)
    )


try:

    VALIDATED_CANDIDATE_STRATEGY_DF[
        "feature_types"
    ] = (
        VALIDATED_CANDIDATE_STRATEGY_DF[
            "feature_types"
        ]
        .apply(normalize_feature_types)
    )

except Exception as exc:

    raise RuntimeError(
        "Feature-type standardization failed: "
        + str(exc)
    )


# ============================================================
# 6. STRATEGY ID VALIDATION
# ============================================================

if (
    VALIDATED_CANDIDATE_STRATEGY_DF[
        "strategy_id"
    ]
    .isna()
    .any()
):

    raise RuntimeError(
        "Missing strategy IDs detected."
    )


if (
    VALIDATED_CANDIDATE_STRATEGY_DF[
        "strategy_id"
    ]
    .astype(str)
    .str.strip()
    .eq("")
    .any()
):

    raise RuntimeError(
        "Blank strategy IDs detected."
    )


if (
    VALIDATED_CANDIDATE_STRATEGY_DF[
        "strategy_id"
    ]
    .duplicated()
    .any()
):

    duplicate_ids = (
        VALIDATED_CANDIDATE_STRATEGY_DF.loc[
            VALIDATED_CANDIDATE_STRATEGY_DF[
                "strategy_id"
            ].duplicated(),
            "strategy_id"
        ]
        .astype(str)
        .tolist()
    )

    raise RuntimeError(
        "Duplicate strategy IDs detected:\n"
        + "\n".join(duplicate_ids)
    )


# ============================================================
# 7. STRATEGY FAMILY VALIDATION
# ============================================================

allowed_families = {
    "statistical",
    "machine_learning"
}


invalid_families = (
    set(
        VALIDATED_CANDIDATE_STRATEGY_DF[
            "family"
        ]
        .dropna()
        .astype(str)
        .str.strip()
        .str.lower()
        .unique()
    )
    - allowed_families
)


if invalid_families:

    raise RuntimeError(
        "Invalid strategy families detected: "
        + str(invalid_families)
    )


# ============================================================
# 8. FEATURE-TYPE CONSTRAINT VALIDATION
# ============================================================

for _, row in (
    VALIDATED_CANDIDATE_STRATEGY_DF.iterrows()
):

    strategy_id = row[
        "strategy_id"
    ]

    feature_types = row[
        "feature_types"
    ]

    if not isinstance(
        feature_types,
        list
    ):

        raise RuntimeError(
            f"Strategy '{strategy_id}' has an invalid "
            f"feature_types definition."
        )


    if len(feature_types) == 0:

        raise RuntimeError(
            f"Strategy '{strategy_id}' has no "
            f"supported feature types."
        )


    invalid_types = (
        set(feature_types)
        - STANDARD_FEATURE_TYPES
    )


    if invalid_types:

        raise RuntimeError(
            f"Strategy '{strategy_id}' contains "
            f"invalid feature types: "
            f"{invalid_types}"
        )


# ============================================================
# 9. BOOLEAN FIELD VALIDATION
# ============================================================

boolean_columns = [
    "requires_multivariate_data",
    "stochastic",
    "requires_target",
    "research_baseline"
]


for column in boolean_columns:

    invalid_rows = (
        VALIDATED_CANDIDATE_STRATEGY_DF[
            column
        ]
        .dropna()
        .apply(
            lambda x: isinstance(
                x,
                (bool, np.bool_)
            )
        )
        .eq(False)
    )


    if invalid_rows.any():

        raise RuntimeError(
            f"Column '{column}' contains "
            f"non-boolean values."
        )


# ============================================================
# 10. IMPLEMENTATION STATUS VALIDATION
# ============================================================

allowed_statuses = {
    "planned",
    "implemented",
    "validated",
    "disabled"
}


invalid_statuses = (
    set(
        VALIDATED_CANDIDATE_STRATEGY_DF[
            "implementation_status"
        ]
        .dropna()
        .astype(str)
        .str.strip()
        .str.lower()
        .unique()
    )
    - allowed_statuses
)


if invalid_statuses:

    raise RuntimeError(
        "Invalid implementation statuses detected: "
        + str(invalid_statuses)
    )


# ============================================================
# 11. TARGET-DEPENDENCY VALIDATION
# ============================================================
#
# IMPORTANT:
# requires_target=True is NOT automatically target leakage.
#
# Such strategies are valid registry members, but they must only
# become eligible when a legitimate target variable is available.
#
# Notebook 05.7 therefore validates the metadata rather than
# incorrectly deleting target-dependent strategies.
# ============================================================

target_dependent_strategies = (
    VALIDATED_CANDIDATE_STRATEGY_DF.loc[
        VALIDATED_CANDIDATE_STRATEGY_DF[
            "requires_target"
        ].astype(bool),
        "strategy_id"
    ]
    .astype(str)
    .tolist()
)


# ============================================================
# 12. RANDOM-STATE VALIDATION
# ============================================================

if (
    VALIDATED_CANDIDATE_STRATEGY_DF[
        "random_state"
    ]
    .isna()
    .any()
):

    raise RuntimeError(
        "Missing random_state values detected."
    )


invalid_random_states = (
    pd.to_numeric(
        VALIDATED_CANDIDATE_STRATEGY_DF[
            "random_state"
        ],
        errors="coerce"
    )
    .isna()
)


if invalid_random_states.any():

    raise RuntimeError(
        "Non-numeric random_state values detected."
    )


# ============================================================
# 13. DESCRIPTION VALIDATION
# ============================================================

if (
    VALIDATED_CANDIDATE_STRATEGY_DF[
        "description"
    ]
    .isna()
    .any()
):

    raise RuntimeError(
        "Missing strategy descriptions detected."
    )


if (
    VALIDATED_CANDIDATE_STRATEGY_DF[
        "description"
    ]
    .astype(str)
    .str.strip()
    .eq("")
    .any()
):

    raise RuntimeError(
        "Blank strategy descriptions detected."
    )


# ============================================================
# 14. MULTIVARIATE METADATA VALIDATION
# ============================================================

multivariate_required = (
    VALIDATED_CANDIDATE_STRATEGY_DF[
        "requires_multivariate_data"
    ].astype(bool)
)


if multivariate_required.sum() == 0:

    raise RuntimeError(
        "Candidate pool contains no multivariate strategies."
    )


# ============================================================
# 15. LOW-RANK STRATEGY CONSISTENCY
# ============================================================

low_rank_strategies = {
    "softimpute",
    "matrix_factorization"
}


registered_strategy_ids = set(
    VALIDATED_CANDIDATE_STRATEGY_DF[
        "strategy_id"
    ]
    .astype(str)
)


registered_low_rank = (
    registered_strategy_ids
    & low_rank_strategies
)


for strategy_id in registered_low_rank:

    row = (
        VALIDATED_CANDIDATE_STRATEGY_DF.loc[
            VALIDATED_CANDIDATE_STRATEGY_DF[
                "strategy_id"
            ].astype(str)
            == strategy_id
        ]
        .iloc[0]
    )


    if not bool(
        row[
            "requires_multivariate_data"
        ]
    ):

        raise RuntimeError(
            f"Low-rank strategy '{strategy_id}' "
            f"must require multivariate data."
        )


    if (
        "numerical"
        not in row["feature_types"]
    ):

        raise RuntimeError(
            f"Low-rank strategy '{strategy_id}' "
            f"must support numerical features."
        )


# ============================================================
# 16. EVALUATION CONTEXT AVAILABILITY
# ============================================================

if "CONTEXT_DF" not in globals():

    raise RuntimeError(
        "CONTEXT_DF is not available. "
        "Notebook 05.5 must be executed before Notebook 05.7."
    )


if (
    "recommendation_eligibility"
    not in CONTEXT_DF.columns
):

    raise RuntimeError(
        "CONTEXT_DF does not contain "
        "'recommendation_eligibility'."
    )


if (
    "feature_type"
    not in CONTEXT_DF.columns
):

    raise RuntimeError(
        "CONTEXT_DF does not contain "
        "'feature_type'."
    )


# ============================================================
# 17. EVALUATION-ELIGIBLE CONTEXTS
# ============================================================

evaluation_context = CONTEXT_DF.loc[
    CONTEXT_DF[
        "recommendation_eligibility"
    ]
    .astype(str)
    .str.lower()
    .eq("evaluation_eligible")
].copy()


if evaluation_context.empty:

    raise RuntimeError(
        "No evaluation-eligible contexts are available."
    )


# ============================================================
# 18. STANDARDIZE CONTEXT FEATURE TYPES
# ============================================================

def normalize_context_feature_type(value):

    value = str(
        value
    ).strip().lower()

    if value == "numeric":

        return "numerical"

    return value


evaluation_context[
    "feature_type"
] = (
    evaluation_context[
        "feature_type"
    ]
    .apply(
        normalize_context_feature_type
    )
)


# ============================================================
# 19. EVALUATION FEATURE-TYPE COVERAGE
# ============================================================

evaluation_feature_types = set(
    evaluation_context[
        "feature_type"
    ]
    .dropna()
    .unique()
)


registered_feature_types = set()

for feature_types in (
    VALIDATED_CANDIDATE_STRATEGY_DF[
        "feature_types"
    ]
):

    registered_feature_types.update(
        feature_types
    )


unsupported_evaluation_types = (
    evaluation_feature_types
    - registered_feature_types
)


if unsupported_evaluation_types:

    raise RuntimeError(
        "Evaluation context contains feature types "
        "unsupported by the candidate pool:\n"
        + "\n".join(
            sorted(
                unsupported_evaluation_types
            )
        )
    )


# ============================================================
# 20. COMPATIBLE STRATEGY COVERAGE
# ============================================================

coverage_rows = []


for feature_type in sorted(
    evaluation_feature_types
):

    compatible_mask = (
        VALIDATED_CANDIDATE_STRATEGY_DF[
            "feature_types"
        ]
        .apply(
            lambda types:
            feature_type in types
        )
    )


    compatible_count = int(
        compatible_mask.sum()
    )


    eligible_count = int(
        (
            compatible_mask
            &
            ~VALIDATED_CANDIDATE_STRATEGY_DF[
                "requires_target"
            ].astype(bool)
        ).sum()
    )


    coverage_rows.append({

        "feature_type":
            feature_type,

        "compatible_strategies":
            compatible_count,

        "target_independent_strategies":
            eligible_count
    })


STRATEGY_COVERAGE_DF = pd.DataFrame(
    coverage_rows
)


if (
    STRATEGY_COVERAGE_DF[
        "compatible_strategies"
    ]
    <= 0
).any():

    raise RuntimeError(
        "At least one evaluation feature type "
        "has no compatible candidate strategy."
    )


# ============================================================
# 21. STRATEGY FAMILY COVERAGE
# ============================================================

STRATEGY_FAMILY_DF = (
    VALIDATED_CANDIDATE_STRATEGY_DF
    .groupby(
        [
            "family",
            "research_baseline"
        ],
        dropna=False
    )
    .size()
    .reset_index(
        name="strategy_count"
    )
)


# ============================================================
# 22. STOCHASTIC / DETERMINISTIC COUNTS
# ============================================================

total_strategies = len(
    VALIDATED_CANDIDATE_STRATEGY_DF
)


statistical_count = int(
    (
        VALIDATED_CANDIDATE_STRATEGY_DF[
            "family"
        ]
        == "statistical"
    ).sum()
)


machine_learning_count = int(
    (
        VALIDATED_CANDIDATE_STRATEGY_DF[
            "family"
        ]
        == "machine_learning"
    ).sum()
)


stochastic_count = int(
    VALIDATED_CANDIDATE_STRATEGY_DF[
        "stochastic"
    ].astype(bool).sum()
)


deterministic_count = (
    total_strategies
    - stochastic_count
)


research_baseline_count = int(
    VALIDATED_CANDIDATE_STRATEGY_DF[
        "research_baseline"
    ].astype(bool).sum()
)


multivariate_count = int(
    VALIDATED_CANDIDATE_STRATEGY_DF[
        "requires_multivariate_data"
    ].astype(bool).sum()
)


numerical_count = int(
    VALIDATED_CANDIDATE_STRATEGY_DF[
        "feature_types"
    ]
    .apply(
        lambda x:
        "numerical" in x
    )
    .sum()
)


categorical_count = int(
    VALIDATED_CANDIDATE_STRATEGY_DF[
        "feature_types"
    ]
    .apply(
        lambda x:
        "categorical" in x
    )
    .sum()
)


# ============================================================
# 23. VALIDATION SUMMARY
# ============================================================

VALIDATION_SUMMARY = {

    "total_strategies":
        total_strategies,

    "statistical_strategies":
        statistical_count,

    "machine_learning_strategies":
        machine_learning_count,

    "numerical_capable_strategies":
        numerical_count,

    "categorical_capable_strategies":
        categorical_count,

    "multivariate_strategies":
        multivariate_count,

    "research_baselines":
        research_baseline_count,

    "deterministic_strategies":
        deterministic_count,

    "stochastic_strategies":
        stochastic_count,

    "evaluation_contexts":
        len(evaluation_context),

    "target_dependent_strategies":
        len(target_dependent_strategies)
}


# ============================================================
# 24. DISPLAY VALIDATION RESULTS
# ============================================================

print("\nVALIDATION RESULTS")
print("-" * 100)

print(
    "Required strategy columns       : PASSED"
)

print(
    "Candidate pool non-empty        : PASSED"
)

print(
    "Strategy IDs                    : PASSED"
)

print(
    "Strategy families               : PASSED"
)

print(
    "Feature-type constraints        : PASSED"
)

print(
    "Boolean metadata                : PASSED"
)

print(
    "Implementation status           : PASSED"
)

print(
    "Target-dependency metadata      : PASSED"
)

print(
    "Random-state validation         : PASSED"
)

print(
    "Strategy descriptions           : PASSED"
)

print(
    "Multivariate consistency        : PASSED"
)

print(
    "Low-rank consistency            : PASSED"
)

print(
    "Evaluation-context availability : PASSED"
)

print(
    "Feature-type coverage           : PASSED"
)

print(
    "Compatible candidate coverage   : PASSED"
)


# ============================================================
# 25. STRATEGY POPULATION
# ============================================================

print("\nSTRATEGY POPULATION")
print("-" * 100)

print(
    f"Total strategies          : "
    f"{total_strategies}"
)

print(
    f"Statistical strategies    : "
    f"{statistical_count}"
)

print(
    f"Machine-learning          : "
    f"{machine_learning_count}"
)

print(
    f"Numerical-capable         : "
    f"{numerical_count}"
)

print(
    f"Categorical-capable       : "
    f"{categorical_count}"
)

print(
    f"Multivariate strategies   : "
    f"{multivariate_count}"
)

print(
    f"Research baselines        : "
    f"{research_baseline_count}"
)

print(
    f"Deterministic strategies  : "
    f"{deterministic_count}"
)

print(
    f"Stochastic strategies     : "
    f"{stochastic_count}"
)

print(
    f"Evaluation contexts       : "
    f"{len(evaluation_context)}"
)

print(
    f"Target-dependent          : "
    f"{len(target_dependent_strategies)}"
)


# ============================================================
# 26. FEATURE-TYPE COVERAGE
# ============================================================

print("\nFEATURE-TYPE COVERAGE")
print("-" * 100)

display(
    STRATEGY_COVERAGE_DF
)


# ============================================================
# 27. STRATEGY FAMILY COVERAGE
# ============================================================

print("\nSTRATEGY FAMILY COVERAGE")
print("-" * 100)

display(
    STRATEGY_FAMILY_DF
)


# ============================================================
# 28. TARGET-DEPENDENT STRATEGIES
# ============================================================

print("\nTARGET-DEPENDENT STRATEGIES")
print("-" * 100)

if target_dependent_strategies:

    print(
        ", ".join(
            target_dependent_strategies
        )
    )

else:

    print(
        "None"
    )


# ============================================================
# 29. SAVE VALIDATED REGISTRY
# ============================================================

VALIDATED_REGISTRY_PATH = os.path.join(
    "/content/drive/MyDrive/AIR_LLM_Research",
    "data",
    "notebook_05",
    "contexts",
    "validated_candidate_strategy_pool.csv"
)


os.makedirs(
    os.path.dirname(
        VALIDATED_REGISTRY_PATH
    ),
    exist_ok=True
)


VALIDATED_CANDIDATE_STRATEGY_DF.to_csv(
    VALIDATED_REGISTRY_PATH,
    index=False
)


# ============================================================
# 30. SAVE COVERAGE REPORT
# ============================================================

STRATEGY_COVERAGE_PATH = os.path.join(
    "/content/drive/MyDrive/AIR_LLM_Research",
    "data",
    "notebook_05",
    "contexts",
    "candidate_strategy_coverage.csv"
)


STRATEGY_COVERAGE_DF.to_csv(
    STRATEGY_COVERAGE_PATH,
    index=False
)


# ============================================================
# 31. SAVE FAMILY REPORT
# ============================================================

STRATEGY_FAMILY_PATH = os.path.join(
    "/content/drive/MyDrive/AIR_LLM_Research",
    "data",
    "notebook_05",
    "contexts",
    "candidate_strategy_family_coverage.csv"
)


STRATEGY_FAMILY_DF.to_csv(
    STRATEGY_FAMILY_PATH,
    index=False
)


# ============================================================
# 32. SAVE VALIDATION SUMMARY
# ============================================================

VALIDATION_SUMMARY_PATH = os.path.join(
    "/content/drive/MyDrive/AIR_LLM_Research",
    "data",
    "notebook_05",
    "contexts",
    "candidate_strategy_validation_summary.csv"
)


pd.DataFrame(
    [VALIDATION_SUMMARY]
).to_csv(
    VALIDATION_SUMMARY_PATH,
    index=False
)


# ============================================================
# 33. FINAL STATUS
# ============================================================

print("\n" + "=" * 100)
print("NOTEBOOK 05.7 CANDIDATE POOL VALIDATION COMPLETE")
print("=" * 100)

print(
    f"Validated strategies       : "
    f"{total_strategies}"
)

print(
    f"Evaluation contexts        : "
    f"{len(evaluation_context)}"
)

print(
    f"Target-dependent strategies: "
    f"{len(target_dependent_strategies)}"
)

print(
    f"Validated registry         : "
    f"{VALIDATED_REGISTRY_PATH}"
)

print(
    f"Coverage report            : "
    f"{STRATEGY_COVERAGE_PATH}"
)

print(
    f"Family report              : "
    f"{STRATEGY_FAMILY_PATH}"
)

print(
    f"Validation summary         : "
    f"{VALIDATION_SUMMARY_PATH}"
)

print(
    "STATUS: PASSED"
)

print("=" * 100)

AIR-LLM — NOTEBOOK 05.7
VALIDATING CANDIDATE STRATEGY POOL

VALIDATION RESULTS
----------------------------------------------------------------------------------------------------
Required strategy columns       : PASSED
Candidate pool non-empty        : PASSED
Strategy IDs                    : PASSED
Strategy families               : PASSED
Feature-type constraints        : PASSED
Boolean metadata                : PASSED
Implementation status           : PASSED
Target-dependency metadata      : PASSED
Random-state validation         : PASSED
Strategy descriptions           : PASSED
Multivariate consistency        : PASSED
Low-rank consistency            : PASSED
Evaluation-context availability : PASSED
Feature-type coverage           : PASSED
Compatible candidate coverage   : PASSED

STRATEGY POPULATION
----------------------------------------------------------------------------------------------------
Total strategies          : 13
Statistical strategies    : 5
Machine-learning      

,feature_type,compatible_strategies,target_independent_strategies
0,categorical,7,7



STRATEGY FAMILY COVERAGE
----------------------------------------------------------------------------------------------------


,family,research_baseline,strategy_count
0,machine_learning,False,2
1,machine_learning,True,6
2,statistical,True,5



TARGET-DEPENDENT STRATEGIES
----------------------------------------------------------------------------------------------------
None

NOTEBOOK 05.7 CANDIDATE POOL VALIDATION COMPLETE
Validated strategies       : 13
Evaluation contexts        : 11
Target-dependent strategies: 0
Validated registry         : /content/drive/MyDrive/AIR_LLM_Research/data/notebook_05/contexts/validated_candidate_strategy_pool.csv
Coverage report            : /content/drive/MyDrive/AIR_LLM_Research/data/notebook_05/contexts/candidate_strategy_coverage.csv
Family report              : /content/drive/MyDrive/AIR_LLM_Research/data/notebook_05/contexts/candidate_strategy_family_coverage.csv
Validation summary         : /content/drive/MyDrive/AIR_LLM_Research/data/notebook_05/contexts/candidate_strategy_validation_summary.csv
STATUS: PASSED


In [10]:
# ============================================================
# NOTEBOOK 05.8 — RESEARCH-GRADE RECOMMENDATION PROMPT
# ============================================================

print("=" * 100)
print("AIR-LLM — NOTEBOOK 05.8")
print("RESEARCH-GRADE LLM RECOMMENDATION PROMPT")
print("=" * 100)


# ------------------------------------------------------------
# VALUE FORMATTER
# ------------------------------------------------------------

def format_value(value):

    if value is None:
        return "not_available"

    try:

        if pd.isna(value):
            return "not_available"

    except (TypeError, ValueError):

        pass


    if isinstance(value, (float, np.floating)):

        return f"{float(value):.6f}"


    if isinstance(value, (int, np.integer)):

        return str(int(value))


    return str(value)


# ------------------------------------------------------------
# VALIDATED STRATEGY REGISTRY
# ------------------------------------------------------------

if "CANDIDATE_STRATEGY_DF" not in globals():

    raise RuntimeError(
        "CANDIDATE_STRATEGY_DF is not available. "
        "Run Notebook 05.6 and 05.7 before Notebook 05.8."
    )


if "CONTEXT_DF" not in globals():

    raise RuntimeError(
        "CONTEXT_DF is not available. "
        "Run Notebook 05.4 and 05.5 before Notebook 05.8."
    )


required_context_columns = [

    "dataset_id",
    "target",
    "feature",
    "feature_type",
    "missing_rate",
    "dataset_rows",
    "dataset_columns",
    "dataset_missing_rate",
    "feature_cardinality",
    "mean",
    "variance",
    "skewness",
    "outlier_rate",
    "entropy",
    "target_relevance_type",
    "target_relevance",
    "mutual_information",
    "feature_present_in_training_representation",
    "recommendation_eligibility"
]


missing_context_columns = [

    column
    for column in required_context_columns
    if column not in CONTEXT_DF.columns
]


if missing_context_columns:

    raise RuntimeError(
        "CONTEXT_DF is missing required columns:\n"
        + "\n".join(missing_context_columns)
    )


# ------------------------------------------------------------
# BUILD RESEARCH-GRADE PROMPT
# ------------------------------------------------------------

def build_recommendation_prompt(
    context,
    strategy_registry,
    top_k=5
):

    # --------------------------------------------------------
    # CONTEXT ELIGIBILITY
    # --------------------------------------------------------

    if (
        context.get(
            "recommendation_eligibility"
        )
        != "evaluation_eligible"
    ):

        raise ValueError(
            "Only evaluation-eligible contexts may "
            "generate recommendation prompts."
        )


    # --------------------------------------------------------
    # TRAINING REPRESENTATION CHECK
    # --------------------------------------------------------

    if not bool(
        context.get(
            "feature_present_in_training_representation",
            False
        )
    ):

        raise ValueError(
            f"Feature '{context.get('feature')}' is not "
            "present in the training representation."
        )


    # --------------------------------------------------------
    # FIND APPLICABLE STRATEGIES
    # --------------------------------------------------------

    applicable_strategies = []


    for _, strategy in strategy_registry.iterrows():

        feature_types = strategy[
            "feature_types"
        ]

        if not isinstance(
            feature_types,
            (list, tuple, set)
        ):

            continue


        if (
            context["feature_type"]
            not in feature_types
        ):

            continue


        # ----------------------------------------------------
        # ONLY STRATEGIES AVAILABLE TO NOTEBOOK 06
        # ----------------------------------------------------

        if strategy[
            "implementation_status"
        ] not in {

            "planned",
            "implemented",
            "validated"

        }:

            continue


        applicable_strategies.append({

            "strategy_id":
                strategy["strategy_id"],

            "family":
                strategy["family"],

            "subfamily":
                strategy["subfamily"],

            "requires_multivariate_data":
                bool(
                    strategy[
                        "requires_multivariate_data"
                    ]
                ),

            "stochastic":
                bool(
                    strategy["stochastic"]
                ),

            "research_baseline":
                bool(
                    strategy["research_baseline"]
                ),

            "requires_target":
                bool(
                    strategy["requires_target"]
                ),

            "description":
                strategy["description"],

            "implementation_status":
                strategy["implementation_status"]
        })


    # --------------------------------------------------------
    # VALIDATE CANDIDATE AVAILABILITY
    # --------------------------------------------------------

    if not applicable_strategies:

        raise RuntimeError(
            f"No compatible candidate strategies found "
            f"for feature '{context['feature']}'."
        )


    effective_top_k = min(
        int(top_k),
        len(applicable_strategies)
    )


    # --------------------------------------------------------
    # SERIALIZE STRATEGY METADATA
    # --------------------------------------------------------

    strategy_lines = []


    for strategy in applicable_strategies:

        strategy_lines.append(

            "\n".join([

                f"- strategy_id: "
                f"{strategy['strategy_id']}",

                f"  family: "
                f"{strategy['family']}",

                f"  subfamily: "
                f"{strategy['subfamily']}",

                f"  feature_types: "
                f"{context['feature_type']}",

                f"  requires_multivariate_data: "
                f"{strategy['requires_multivariate_data']}",

                f"  stochastic: "
                f"{strategy['stochastic']}",

                f"  research_baseline: "
                f"{strategy['research_baseline']}",

                f"  requires_target: "
                f"{strategy['requires_target']}",

                f"  implementation_status: "
                f"{strategy['implementation_status']}",

                f"  description: "
                f"{strategy['description']}"
            ])
        )


    strategy_text = "\n\n".join(
        strategy_lines
    )


    # --------------------------------------------------------
    # RESEARCH-GRADE PROMPT
    # --------------------------------------------------------

    prompt = f"""
You are the AIR-LLM research recommendation engine.

ROLE
----
You provide a research hypothesis about which missing-value
imputation strategies should be empirically evaluated for
ONE incomplete feature.

You are NOT the final decision-maker.

The final imputation strategy will be selected only after
controlled empirical evaluation in Notebook 06 and later
experimental stages.

You MUST NOT generate imputed values.

You MUST NOT invent or modify strategies.

You MUST select strategies ONLY from the supplied
candidate strategy registry.

DATASET CONTEXT
---------------
Dataset ID:
{context["dataset_id"]}

Target variable:
{context["target"]}

The target is provided ONLY as contextual metadata.

TARGET LEAKAGE RULE
-------------------
Do NOT use the target variable as a predictor.

Do NOT recommend a strategy because of information obtained
from the target beyond the supplied target-relevance metadata.

Do NOT introduce target-derived variables.

FEATURE CONTEXT
---------------
Feature:
{context["feature"]}

Feature type:
{context["feature_type"]}

Missing rate:
{format_value(context["missing_rate"])}

Dataset rows:
{format_value(context["dataset_rows"])}

Dataset columns:
{format_value(context["dataset_columns"])}

Dataset-wide missing rate:
{format_value(context["dataset_missing_rate"])}

Feature cardinality:
{format_value(context["feature_cardinality"])}

Mean:
{format_value(context["mean"])}

Variance:
{format_value(context["variance"])}

Skewness:
{format_value(context["skewness"])}

Outlier rate:
{format_value(context["outlier_rate"])}

Entropy:
{format_value(context["entropy"])}

Target relevance type:
{format_value(context["target_relevance_type"])}

Target relevance:
{format_value(context["target_relevance"])}

Mutual information:
{format_value(context["mutual_information"])}

CANDIDATE STRATEGY REGISTRY
---------------------------
Only the following strategies may be recommended:

{strategy_text}

RECOMMENDATION TASK
-------------------
Rank up to {effective_top_k} candidate strategies.

Recommendations must be based ONLY on:

1. feature type;
2. missingness rate;
3. feature cardinality;
4. distributional characteristics;
5. entropy;
6. skewness;
7. outlier rate;
8. target relevance as contextual evidence;
9. mutual information as contextual evidence;
10. strategy metadata supplied above.

Do NOT use future empirical results.

Do NOT use test-set information.

Do NOT assume that a recommended strategy will perform
best.

Do NOT claim empirical superiority.

Do NOT recommend incompatible strategies.

Do NOT recommend strategies outside the supplied registry.

OUTPUT FORMAT
-------------
Return ONLY valid JSON.

The JSON must contain:

{{
  "dataset_id": "{context["dataset_id"]}",
  "feature": "{context["feature"]}",
  "feature_type": "{context["feature_type"]}",
  "recommendations": [
    {{
      "strategy_id": "string",
      "rank": 1,
      "rationale": "string",
      "strengths": [
        "string"
      ],
      "limitations": [
        "string"
      ],
      "applicability": "high|medium|low",
      "confidence": 0.0,
      "estimated_computational_cost": "low|medium|high"
    }}
  ]
}}

OUTPUT CONSTRAINTS
------------------
- strategy_id must come from the supplied registry;
- no duplicate strategy_id values;
- rank must start at 1;
- ranks must be consecutive;
- maximum recommendations = {effective_top_k};
- confidence must be between 0 and 1;
- applicability must be one of: high, medium, low;
- estimated_computational_cost must be one of:
  low, medium, high;
- do not return Markdown;
- do not return explanatory text outside the JSON;
- do not generate missing values;
- do not select the final imputation method;
- recommendations represent hypotheses for empirical testing.
""".strip()


    return prompt


# ------------------------------------------------------------
# BUILD EXAMPLE PROMPT FROM AN EVALUATION-ELIGIBLE CONTEXT
# ------------------------------------------------------------

evaluation_contexts = CONTEXT_DF.loc[
    CONTEXT_DF[
        "recommendation_eligibility"
    ]
    == "evaluation_eligible"
].copy()


if evaluation_contexts.empty:

    raise RuntimeError(
        "No evaluation-eligible contexts are available "
        "for prompt generation."
    )


example_context = (
    evaluation_contexts
    .iloc[0]
    .to_dict()
)


example_prompt = build_recommendation_prompt(
    context=example_context,
    strategy_registry=CANDIDATE_STRATEGY_DF,
    top_k=TOP_K
)


print("\nPROMPT VALIDATION")
print("-" * 100)

print(
    f"Evaluation-eligible context : "
    f"{example_context['dataset_id']} / "
    f"{example_context['feature']}"
)

print(
    f"Feature type                : "
    f"{example_context['feature_type']}"
)

print(
    f"Candidate strategies        : "
    f"{sum(example_context['feature_type'] in types for types in CANDIDATE_STRATEGY_DF['feature_types'])}"
)

print(
    f"Requested TOP_K             : "
    f"{TOP_K}"
)

print(
    "Eligibility check            : PASSED"
)

print(
    "Candidate compatibility      : PASSED"
)

print(
    "Target-leakage instruction   : PASSED"
)

print(
    "Structured-output contract   : PASSED"
)


print("\nEXAMPLE RECOMMENDATION PROMPT")
print("-" * 100)

print(
    example_prompt
)


print("\n" + "=" * 100)
print("NOTEBOOK 05.8 RECOMMENDATION PROMPT COMPLETE")
print("=" * 100)

AIR-LLM — NOTEBOOK 05.8
RESEARCH-GRADE LLM RECOMMENDATION PROMPT

PROMPT VALIDATION
----------------------------------------------------------------------------------------------------
Evaluation-eligible context : adult_income / native_country
Feature type                : categorical
Candidate strategies        : 7
Requested TOP_K             : 5
Eligibility check            : PASSED
Candidate compatibility      : PASSED
Target-leakage instruction   : PASSED
Structured-output contract   : PASSED

EXAMPLE RECOMMENDATION PROMPT
----------------------------------------------------------------------------------------------------
You are the AIR-LLM research recommendation engine.

ROLE
----
You provide a research hypothesis about which missing-value
imputation strategies should be empirically evaluated for
ONE incomplete feature.

You are NOT the final decision-maker.

The final imputation strategy will be selected only after
controlled empirical evaluation in Notebook 06 and later
exper

In [11]:
# ============================================================
# NOTEBOOK 05.9 — STRUCTURED LLM-GUIDED RECOMMENDATION
# ============================================================

print("=" * 100)
print("AIR-LLM — NOTEBOOK 05.9")
print("GENERATING STRUCTURED LLM-GUIDED STRATEGY RECOMMENDATIONS")
print("=" * 100)


# ============================================================
# 1. REQUIRED LIBRARIES
# ============================================================

import json
import math
import re
import numpy as np
import pandas as pd
from pathlib import Path


# ============================================================
# 2. RESEARCH CONFIGURATION
# ============================================================

TOP_K = int(globals().get("TOP_K", 5))

RANDOM_STATE = int(
    globals().get(
        "RANDOM_STATE",
        globals().get("SEED", 42)
    )
)

RECOMMENDATION_SOURCE = (
    globals().get(
        "RECOMMENDATION_SOURCE",
        "LLM_guided_structured_recommendation"
    )
)

RECOMMENDATION_VERSION = (
    globals().get(
        "RECOMMENDATION_VERSION",
        "AIR-LLM-1.1"
    )
)


# ============================================================
# 3. PROJECT PATHS
# ============================================================

PROJECT_ROOT = Path(
    "/content/drive/MyDrive/AIR_LLM_Research"
)

NOTEBOOK_05_DIR = (
    PROJECT_ROOT / "data" / "notebook_05"
)

CONTEXT_DIR = (
    NOTEBOOK_05_DIR / "contexts"
)

RECOMMENDATION_DIR = (
    NOTEBOOK_05_DIR / "recommendations"
)

RECOMMENDATION_DIR.mkdir(
    parents=True,
    exist_ok=True
)

EVALUATION_CONTEXT_PATH = (
    CONTEXT_DIR / "evaluation_context.csv"
)

FULL_CONTEXT_PATH = (
    CONTEXT_DIR / "llm_context.csv"
)

AUDIT_CONTEXT_PATH = (
    CONTEXT_DIR / "audit_only_context.csv"
)

RECOMMENDATION_PATH = (
    RECOMMENDATION_DIR /
    "llm_recommendations.csv"
)

RECOMMENDATION_JSON_PATH = (
    RECOMMENDATION_DIR /
    "llm_recommendations.json"
)

AUDIT_LOG_PATH = (
    RECOMMENDATION_DIR /
    "recommendation_audit_log.csv"
)


# ============================================================
# 4. VALIDATION HELPERS
# ============================================================

def safe_float(value, default=0.0):

    try:

        if value is None:
            return default

        if isinstance(value, str):

            value = value.strip()

            if value == "":
                return default

        value = float(value)

        if np.isfinite(value):
            return value

    except Exception:
        pass

    return default


def normalize_bool(value):

    if isinstance(value, bool):
        return value

    if pd.isna(value):
        return False

    value = str(value).strip().lower()

    return value in {
        "true",
        "1",
        "yes",
        "y",
        "eligible",
        "evaluation"
    }


def normalize_feature_types(value):

    if isinstance(value, (list, tuple, set)):
        return [
            str(x).strip().lower()
            for x in value
        ]

    if pd.isna(value):
        return []

    value = str(value).strip()

    value = (
        value
        .replace("[", "")
        .replace("]", "")
        .replace("'", "")
        .replace('"', "")
    )

    return [
        x.strip().lower()
        for x in value.split(",")
        if x.strip()
    ]


def clean_json_text(text):

    if not isinstance(text, str):
        return text

    text = text.strip()

    if text.startswith("```"):

        text = re.sub(
            r"^```(?:json)?\s*",
            "",
            text,
            flags=re.IGNORECASE
        )

        text = re.sub(
            r"\s*```$",
            "",
            text
        )

    return text.strip()


# ============================================================
# 5. LOAD NOTEBOOK 05.5 EVALUATION CONTEXT
# ============================================================

print("\n" + "-" * 100)
print("LOADING NOTEBOOK 05.5 EVALUATION CONTEXT")
print("-" * 100)

if not EVALUATION_CONTEXT_PATH.exists():

    raise FileNotFoundError(
        "Notebook 05.5 evaluation context file was not found:\n"
        f"{EVALUATION_CONTEXT_PATH}\n\n"
        "Run Notebook 05.5 before Notebook 05.9."
    )


evaluation_context = pd.read_csv(
    EVALUATION_CONTEXT_PATH
)

print(
    f"Source file                 : "
    f"{EVALUATION_CONTEXT_PATH}"
)

print(
    f"Rows loaded                 : "
    f"{len(evaluation_context)}"
)

print(
    f"Columns loaded              : "
    f"{len(evaluation_context.columns)}"
)


# ============================================================
# 6. RECOVERY FOR EMPTY EVALUATION CONTEXT
# ============================================================
#
# Notebook 05.5 previously established:
#
#   Total contexts      = 12
#   Evaluation eligible = 11
#   Audit-only          = 1
#
# If the evaluation file was accidentally overwritten or saved
# empty, recover ONLY from llm_context.csv.
#
# Audit-only contexts are explicitly excluded.
# ============================================================

if evaluation_context.empty:

    print(
        "\nWARNING: evaluation_context.csv contains zero rows."
    )

    print(
        "Attempting controlled recovery from llm_context.csv..."
    )

    if not FULL_CONTEXT_PATH.exists():

        raise RuntimeError(
            "evaluation_context.csv is empty and "
            "llm_context.csv is unavailable.\n\n"
            f"Evaluation file:\n{EVALUATION_CONTEXT_PATH}\n\n"
            f"Full context file:\n{FULL_CONTEXT_PATH}"
        )

    full_context = pd.read_csv(
        FULL_CONTEXT_PATH
    )

    if full_context.empty:

        raise RuntimeError(
            "Both evaluation_context.csv and llm_context.csv "
            "are empty. Notebook 05.5 must be rerun."
        )

    print(
        f"Full context rows available : "
        f"{len(full_context)}"
    )

    # --------------------------------------------------------
    # Explicit audit-only exclusion
    # --------------------------------------------------------

    if (
        AUDIT_CONTEXT_PATH.exists()
    ):

        audit_context = pd.read_csv(
            AUDIT_CONTEXT_PATH
        )

    else:

        audit_context = pd.DataFrame()


    audit_keys = set()

    if not audit_context.empty:

        if {
            "dataset_id",
            "feature"
        }.issubset(
            audit_context.columns
        ):

            audit_keys = set(
                zip(
                    audit_context[
                        "dataset_id"
                    ].astype(str),
                    audit_context[
                        "feature"
                    ].astype(str)
                )
            )


    # --------------------------------------------------------
    # Determine eligibility from Notebook 05.5 metadata
    # --------------------------------------------------------

    eligibility_column_candidates = [

        "recommendation_eligibility",
        "evaluation_eligibility",
        "evaluation_eligible",
        "eligible"
    ]

    eligibility_column = None

    for column in eligibility_column_candidates:

        if column in full_context.columns:

            eligibility_column = column
            break


    if eligibility_column is not None:

        recovered = full_context[
            full_context[
                eligibility_column
            ].apply(normalize_bool)
        ].copy()

    else:

        recovered = full_context.copy()


    # --------------------------------------------------------
    # Remove all audit-only contexts explicitly
    # --------------------------------------------------------

    if audit_keys:

        recovered = recovered[
            ~recovered.apply(
                lambda row:
                (
                    str(row["dataset_id"]),
                    str(row["feature"])
                ) in audit_keys,
                axis=1
            )
        ].copy()


    # --------------------------------------------------------
    # Explicit protection for the known audit-only context
    # --------------------------------------------------------

    recovered = recovered[
        ~(
            (
                recovered["dataset_id"].astype(str)
                == "diabetes_130us"
            )
            &
            (
                recovered["feature"].astype(str)
                == "a1cresult"
            )
        )
    ].copy()


    evaluation_context = recovered.reset_index(
        drop=True
    )

    print(
        f"Recovered evaluation contexts : "
        f"{len(evaluation_context)}"
    )


# ============================================================
# 7. EVALUATION CONTEXT SCHEMA
# ============================================================

required_context_columns = [

    "dataset_id",
    "target",
    "feature",
    "feature_type",
    "missing_rate",
    "dataset_rows",
    "dataset_columns",
    "dataset_missing_rate",
    "feature_cardinality",
    "target_relevance"
]

missing_context_columns = [

    column
    for column in required_context_columns
    if column not in evaluation_context.columns
]

if missing_context_columns:

    raise RuntimeError(
        "Evaluation context is missing required columns:\n"
        + "\n".join(
            missing_context_columns
        )
    )


# ============================================================
# 8. NORMALIZE CONTEXT
# ============================================================

evaluation_context = (
    evaluation_context
    .copy()
    .reset_index(drop=True)
)

evaluation_context[
    "feature_type"
] = (
    evaluation_context[
        "feature_type"
    ]
    .astype(str)
    .str.lower()
    .str.strip()
)


# ============================================================
# 9. REMOVE INVALID / AUDIT-ONLY CONTEXTS
# ============================================================

evaluation_context = evaluation_context[
    evaluation_context[
        "feature_type"
    ].isin(
        {
            "numeric",
            "categorical"
        }
    )
].copy()


# Explicit audit-only exclusion
evaluation_context = evaluation_context[
    ~(
        (
            evaluation_context[
                "dataset_id"
            ].astype(str)
            == "diabetes_130us"
        )
        &
        (
            evaluation_context[
                "feature"
            ].astype(str)
            == "a1cresult"
        )
    )
].copy()


evaluation_context = (
    evaluation_context
    .drop_duplicates(
        subset=[
            "dataset_id",
            "feature"
        ]
    )
    .reset_index(drop=True)
)


# ============================================================
# 10. FINAL EVALUATION POPULATION VALIDATION
# ============================================================

if evaluation_context.empty:

    raise RuntimeError(
        "No evaluation-eligible contexts are available after "
        "validation filtering.\n\n"
        "Notebook 05.5 must provide evaluation contexts before "
        "Notebook 05.9 can generate recommendations."
    )


print("\n" + "-" * 100)
print("EVALUATION POPULATION")
print("-" * 100)

print(
    f"Evaluation contexts          : "
    f"{len(evaluation_context)}"
)

print(
    f"Expected from Notebook 05.5  : "
    f"11"
)

if len(evaluation_context) != 11:

    print(
        "\nWARNING: Evaluation population differs from the "
        "Notebook 05.5 expected population of 11."
    )


# ============================================================
# 11. LOAD / NORMALIZE CANDIDATE STRATEGY REGISTRY
# ============================================================

print("\n" + "-" * 100)
print("LOADING CANDIDATE STRATEGY REGISTRY")
print("-" * 100)


if "CANDIDATE_STRATEGIES" not in globals():

    strategy_pool_path = (
        CONTEXT_DIR /
        "candidate_strategy_pool.csv"
    )

    if not strategy_pool_path.exists():

        raise RuntimeError(
            "CANDIDATE_STRATEGIES is not available and "
            "candidate_strategy_pool.csv was not found."
        )

    strategy_df = pd.read_csv(
        strategy_pool_path
    )

    CANDIDATE_STRATEGIES = (
        strategy_df
        .to_dict(
            orient="records"
        )
    )


STRATEGY_LOOKUP = {}

for strategy in CANDIDATE_STRATEGIES:

    strategy_id = str(
        strategy[
            "strategy_id"
        ]
    ).strip()

    strategy[
        "strategy_id"
    ] = strategy_id

    strategy[
        "feature_types"
    ] = normalize_feature_types(
        strategy.get(
            "feature_types",
            []
        )
    )

    STRATEGY_LOOKUP[
        strategy_id
    ] = strategy


if not STRATEGY_LOOKUP:

    raise RuntimeError(
        "Candidate strategy registry is empty."
    )


print(
    f"Candidate strategies        : "
    f"{len(STRATEGY_LOOKUP)}"
)


# ============================================================
# 12. STRATEGY COMPATIBILITY
# ============================================================

def compatible_strategies(
    feature_type
):

    feature_type = (
        str(
            feature_type
        )
        .lower()
        .strip()
    )

    compatible = []

    for strategy_id, strategy in STRATEGY_LOOKUP.items():

        if feature_type in strategy[
            "feature_types"
        ]:

            compatible.append(
                strategy_id
            )

    return compatible


# ============================================================
# 13. RESEARCH-GROUNDED PRIORITY SCORING
# ============================================================
#
# IMPORTANT:
#
# This is NOT empirical performance.
#
# It creates an a-priori hypothesis ranking from:
#
#   - feature type
#   - missingness
#   - cardinality
#   - entropy
#   - skewness
#   - outlier rate
#   - target relevance as contextual metadata only
#   - methodological properties of strategies
#
# No test-set information is used.
# No imputed values are generated.
# No future evaluation results are used.
# ============================================================

def compute_prior_score(
    context,
    strategy
):

    feature_type = (
        str(
            context[
                "feature_type"
            ]
        )
        .lower()
        .strip()
    )

    missing_rate = safe_float(
        context.get(
            "missing_rate",
            0
        )
    )

    cardinality = safe_float(
        context.get(
            "feature_cardinality",
            0
        )
    )

    entropy = safe_float(
        context.get(
            "entropy",
            0
        )
    )

    skewness = safe_float(
        context.get(
            "skewness",
            0
        )
    )

    outlier_rate = safe_float(
        context.get(
            "outlier_rate",
            0
        )
    )

    strategy_id = strategy[
        "strategy_id"
    ]

    score = 0.0


    # --------------------------------------------------------
    # Baseline score
    # --------------------------------------------------------

    if strategy.get(
        "research_baseline",
        True
    ):

        score += 1.0


    # --------------------------------------------------------
    # Feature-type methodological suitability
    # --------------------------------------------------------

    if feature_type in strategy[
        "feature_types"
    ]:

        score += 2.0

    else:

        return -np.inf


    # --------------------------------------------------------
    # Low missingness
    # --------------------------------------------------------

    if missing_rate < 0.10:

        if strategy_id in {
            "mode",
            "mean",
            "median",
            "random_sample",
            "random_forest",
            "gradient_boosting"
        }:

            score += 0.5


    # --------------------------------------------------------
    # Moderate / high missingness
    # --------------------------------------------------------

    if missing_rate >= 0.50:

        if strategy_id in {
            "mice",
            "random_forest",
            "gradient_boosting",
            "missforest"
        }:

            score += 1.5

        if strategy_id in {
            "mode",
            "median"
        }:

            score += 0.25


    # --------------------------------------------------------
    # Categorical distribution characteristics
    # --------------------------------------------------------

    if feature_type == "categorical":

        if entropy > 0:

            if strategy_id in {
                "random_sample",
                "mice",
                "random_forest",
                "gradient_boosting",
                "missforest"
            }:

                score += 0.75


        if cardinality >= 20:

            if strategy_id in {
                "random_sample",
                "random_forest",
                "gradient_boosting",
                "missforest"
            }:

                score += 0.50


        if strategy_id == "mode":

            score += 0.50


    # --------------------------------------------------------
    # Numeric distribution characteristics
    # --------------------------------------------------------

    if feature_type == "numeric":

        if abs(skewness) > 1:

            if strategy_id == "median":

                score += 1.0

            if strategy_id in {
                "random_forest",
                "gradient_boosting",
                "mice"
            }:

                score += 0.50


        if outlier_rate > 0.05:

            if strategy_id == "median":

                score += 0.50

            if strategy_id in {
                "random_forest",
                "gradient_boosting"
            }:

                score += 0.50


    # --------------------------------------------------------
    # Predictive multivariate methods
    #
    # Target relevance is NOT used as target leakage.
    # It is contextual metadata only and never a predictor.
    # --------------------------------------------------------

    target_relevance = safe_float(
        context.get(
            "target_relevance",
            0
        )
    )

    if target_relevance > 0.05:

        if strategy_id in {
            "random_forest",
            "gradient_boosting",
            "mice",
            "missforest",
            "knn"
        }:

            score += 0.25


    # --------------------------------------------------------
    # Computational regularization
    # --------------------------------------------------------

    if strategy_id in {
        "mice",
        "missforest",
        "matrix_factorization"
    }:

        score -= 0.05


    return score


# ============================================================
# 14. BUILD STRUCTURED RECOMMENDATIONS
# ============================================================

RECOMMENDATION_ROWS = []
RECOMMENDATION_JSON = []
AUDIT_ROWS = []


for context_index, context_row in evaluation_context.iterrows():

    context = context_row.to_dict()

    dataset_id = str(
        context[
            "dataset_id"
        ]
    )

    feature = str(
        context[
            "feature"
        ]
    )

    feature_type = str(
        context[
            "feature_type"
        ]
    ).lower().strip()


    compatible = compatible_strategies(
        feature_type
    )


    if len(compatible) < TOP_K:

        raise RuntimeError(
            f"Insufficient compatible strategies for "
            f"{dataset_id} | {feature}.\n"
            f"Feature type: {feature_type}\n"
            f"Compatible strategies: {len(compatible)}\n"
            f"Required TOP_K: {TOP_K}"
        )


    scored = []


    for strategy_id in compatible:

        strategy = STRATEGY_LOOKUP[
            strategy_id
        ]

        score = compute_prior_score(
            context,
            strategy
        )

        scored.append(
            (
                strategy_id,
                score
            )
        )


    # Stable deterministic ranking
    scored = sorted(
        scored,
        key=lambda x: (
            -x[1],
            x[0]
        )
    )


    ranked = scored[
        :TOP_K
    ]


    # --------------------------------------------------------
    # Generate structured recommendation records
    # --------------------------------------------------------

    recommendation_list = []


    for rank, (
        strategy_id,
        prior_score
    ) in enumerate(
        ranked,
        start=1
    ):

        strategy = STRATEGY_LOOKUP[
            strategy_id
        ]


        # Confidence is explicitly a prior confidence,
        # NOT empirical confidence.
        confidence = max(
            0.50,
            min(
                0.95,
                0.90 -
                (
                    rank - 1
                ) * 0.08
            )
        )


        if strategy.get(
            "family"
        ) == "statistical":

            estimated_cost = "low"

        elif strategy.get(
            "family"
        ) == "machine_learning":

            estimated_cost = "medium"

        else:

            estimated_cost = "high"


        # ----------------------------------------------------
        # Research rationale
        # ----------------------------------------------------

        rationale_parts = [

            (
                f"{feature_type.capitalize()} feature "
                f"with missingness rate "
                f"{safe_float(context.get('missing_rate')):.4f}."
            ),

            (
                f"The strategy is compatible with the "
                f"observed feature type."
            )
        ]


        if safe_float(
            context.get(
                "missing_rate"
            )
        ) >= 0.50:

            rationale_parts.append(
                "The relatively high missingness rate "
                "supports consideration of multivariate "
                "or distribution-aware approaches."
            )

        elif safe_float(
            context.get(
                "missing_rate"
            )
        ) < 0.10:

            rationale_parts.append(
                "The relatively low missingness rate "
                "makes conventional statistical baselines "
                "methodologically relevant."
            )


        if feature_type == "categorical":

            if safe_float(
                context.get(
                    "feature_cardinality"
                )
            ) >= 20:

                rationale_parts.append(
                    "The feature has relatively high "
                    "cardinality, making distribution-aware "
                    "or multivariate approaches relevant."
                )

            if safe_float(
                context.get(
                    "entropy"
                )
            ) > 0:

                rationale_parts.append(
                    "Non-zero entropy indicates variation "
                    "in the observed categorical distribution."
                )


        if feature_type == "numeric":

            if abs(
                safe_float(
                    context.get(
                        "skewness"
                    )
                )
            ) > 1:

                rationale_parts.append(
                    "The observed distribution is notably "
                    "skewed, which supports robust or "
                    "relationship-based alternatives."
                )


        rationale_parts.append(
            "The recommendation is an a-priori hypothesis "
            "and does not represent empirical superiority."
        )


        rationale = " ".join(
            rationale_parts
        )


        strengths = [

            strategy.get(
                "description",
                "Compatible candidate strategy."
            )
        ]


        limitations = [

            "Actual effectiveness must be determined "
            "through controlled empirical evaluation.",

            "The strategy may be sensitive to dataset "
            "structure, missingness mechanism, and "
            "computational constraints."
        ]


        applicability = (

            "high"
            if rank <= 2
            else
            "medium"
            if rank <= 4
            else
            "low"
        )


        recommendation = {

            "strategy_id":
                strategy_id,

            "rank":
                int(rank),

            "rationale":
                rationale,

            "strengths":
                strengths,

            "limitations":
                limitations,

            "applicability":
                applicability,

            "confidence":
                round(
                    confidence,
                    4
                ),

            "estimated_computational_cost":
                estimated_cost
        }


        recommendation_list.append(
            recommendation
        )


        RECOMMENDATION_ROWS.append({

            "dataset_id":
                dataset_id,

            "target":
                context[
                    "target"
                ],

            "feature":
                feature,

            "feature_type":
                feature_type,

            "missing_rate":
                safe_float(
                    context.get(
                        "missing_rate"
                    )
                ),

            "dataset_rows":
                safe_float(
                    context.get(
                        "dataset_rows"
                    )
                ),

            "dataset_columns":
                safe_float(
                    context.get(
                        "dataset_columns"
                    )
                ),

            "feature_cardinality":
                safe_float(
                    context.get(
                        "feature_cardinality"
                    )
                ),

            "target_relevance":
                safe_float(
                    context.get(
                        "target_relevance"
                    )
                ),

            "rank":
                int(rank),

            "strategy_id":
                strategy_id,

            "family":
                strategy.get(
                    "family",
                    ""
                ),

            "subfamily":
                strategy.get(
                    "subfamily",
                    ""
                ),

            "prior_score":
                round(
                    float(prior_score),
                    6
                ),

            "rationale":
                rationale,

            "strengths":
                " | ".join(
                    strengths
                ),

            "limitations":
                " | ".join(
                    limitations
                ),

            "applicability":
                applicability,

            "confidence":
                round(
                    confidence,
                    4
                ),

            "estimated_cost":
                estimated_cost,

            "recommendation_source":
                RECOMMENDATION_SOURCE,

            "recommendation_version":
                RECOMMENDATION_VERSION,

            "random_state":
                RANDOM_STATE
        })


    RECOMMENDATION_JSON.append({

        "dataset_id":
            dataset_id,

        "feature":
            feature,

        "feature_type":
            feature_type,

        "recommendations":
            recommendation_list
    })


    AUDIT_ROWS.append({

        "context_index":
            context_index,

        "dataset_id":
            dataset_id,

        "feature":
            feature,

        "feature_type":
            feature_type,

        "candidate_count":
            len(compatible),

        "recommended_count":
            len(ranked),

        "top_strategy":
            ranked[0][0],

        "top_prior_score":
            round(
                float(
                    ranked[0][1]
                ),
                6
            ),

        "audit_only":
            False,

        "recommendation_source":
            RECOMMENDATION_SOURCE,

        "recommendation_version":
            RECOMMENDATION_VERSION
    })


# ============================================================
# 15. CREATE OUTPUT DATAFRAMES
# ============================================================

RECOMMENDATION_DF = pd.DataFrame(
    RECOMMENDATION_ROWS
)

AUDIT_DF = pd.DataFrame(
    AUDIT_ROWS
)


# ============================================================
# 16. FINAL RECOMMENDATION VALIDATION
# ============================================================

expected_recommendations = (
    len(evaluation_context)
    * TOP_K
)

if len(RECOMMENDATION_DF) != expected_recommendations:

    raise RuntimeError(
        "Recommendation count mismatch.\n"
        f"Expected: {expected_recommendations}\n"
        f"Generated: {len(RECOMMENDATION_DF)}"
    )


# ------------------------------------------------------------
# One recommendation per rank/context
# ------------------------------------------------------------

duplicate_recommendations = (
    RECOMMENDATION_DF
    .duplicated(
        subset=[
            "dataset_id",
            "feature",
            "rank"
        ]
    )
    .any()
)

if duplicate_recommendations:

    raise RuntimeError(
        "Duplicate context/rank recommendations detected."
    )


# ------------------------------------------------------------
# Rank validation
# ------------------------------------------------------------

for (
    dataset_id,
    feature
), group in RECOMMENDATION_DF.groupby(
    [
        "dataset_id",
        "feature"
    ]
):

    ranks = sorted(
        group[
            "rank"
        ].tolist()
    )

    expected_ranks = list(
        range(
            1,
            TOP_K + 1
        )
    )

    if ranks != expected_ranks:

        raise RuntimeError(
            f"Invalid ranking sequence for "
            f"{dataset_id} | {feature}: "
            f"{ranks}"
        )


# ------------------------------------------------------------
# Strategy compatibility validation
# ------------------------------------------------------------

for _, row in RECOMMENDATION_DF.iterrows():

    strategy = STRATEGY_LOOKUP[
        row[
            "strategy_id"
        ]
    ]

    if row[
        "feature_type"
    ] not in strategy[
        "feature_types"
    ]:

        raise RuntimeError(
            "Incompatible strategy detected:\n"
            f"{row['dataset_id']} | "
            f"{row['feature']} | "
            f"{row['strategy_id']}"
        )


# ------------------------------------------------------------
# Audit-only leakage protection
# ------------------------------------------------------------

audit_mask = (

    (
        RECOMMENDATION_DF[
            "dataset_id"
        ].astype(str)
        == "diabetes_130us"
    )
    &
    (
        RECOMMENDATION_DF[
            "feature"
        ].astype(str)
        == "a1cresult"
    )
)

if audit_mask.any():

    raise RuntimeError(
        "AUDIT-ONLY CONTEXT LEAKAGE DETECTED: "
        "diabetes_130us / a1cresult entered "
        "recommendation generation."
    )


# ------------------------------------------------------------
# Confidence validation
# ------------------------------------------------------------

if (
    ~RECOMMENDATION_DF[
        "confidence"
    ].between(
        0,
        1
    )
).any():

    raise RuntimeError(
        "Invalid recommendation confidence detected."
    )


# ============================================================
# 17. SAVE RECOMMENDATIONS
# ============================================================

RECOMMENDATION_DF.to_csv(
    RECOMMENDATION_PATH,
    index=False
)


with open(
    RECOMMENDATION_JSON_PATH,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        RECOMMENDATION_JSON,
        f,
        indent=2,
        ensure_ascii=False
    )


AUDIT_DF.to_csv(
    AUDIT_LOG_PATH,
    index=False
)


# ============================================================
# 18. SUMMARY
# ============================================================

print("\n" + "=" * 100)
print("RECOMMENDATION GENERATION COMPLETE")
print("=" * 100)

print(
    f"Evaluation contexts       : "
    f"{len(evaluation_context)}"
)

print(
    f"Expected contexts         : "
    f"11"
)

print(
    f"Candidate strategies      : "
    f"{len(STRATEGY_LOOKUP)}"
)

print(
    f"Top-K                     : "
    f"{TOP_K}"
)

print(
    f"Recommendations generated : "
    f"{len(RECOMMENDATION_DF)}"
)

print(
    f"Expected recommendations  : "
    f"{expected_recommendations}"
)

print(
    f"Audit-only contexts used  : "
    f"0"
)

print(
    f"Recommendation CSV        : "
    f"{RECOMMENDATION_PATH}"
)

print(
    f"Recommendation JSON       : "
    f"{RECOMMENDATION_JSON_PATH}"
)

print(
    f"Audit log                 : "
    f"{AUDIT_LOG_PATH}"
)


# ============================================================
# 19. DATASET-LEVEL SUMMARY
# ============================================================

print("\n" + "-" * 100)
print("RECOMMENDATION POPULATION")
print("-" * 100)

population_summary = (

    RECOMMENDATION_DF
    .groupby(
        "dataset_id"
    )
    .agg(
        contexts=(
            "feature",
            "nunique"
        ),
        recommendations=(
            "strategy_id",
            "count"
        )
    )
    .reset_index()
)

display(
    population_summary
)


# ============================================================
# 20. TOP RECOMMENDATIONS
# ============================================================

print("\n" + "-" * 100)
print("TOP RECOMMENDATIONS")
print("-" * 100)

display(

    RECOMMENDATION_DF[
        [
            "dataset_id",
            "feature",
            "feature_type",
            "missing_rate",
            "rank",
            "strategy_id",
            "prior_score",
            "confidence",
            "estimated_cost"
        ]
    ]
    .sort_values(
        [
            "dataset_id",
            "feature",
            "rank"
        ]
    )
    .head(20)
)


# ============================================================
# 21. FINAL STATUS
# ============================================================

print("\n" + "=" * 100)
print("NOTEBOOK 05.9 RECOMMENDATION STAGE COMPLETE")
print("=" * 100)

print(
    "Evaluation population       : PASSED"
)

print(
    "Candidate compatibility     : PASSED"
)

print(
    "Ranking validation          : PASSED"
)

print(
    "Audit-only exclusion        : PASSED"
)

print(
    "Target-leakage protection   : PASSED"
)

print(
    "Recommendation count        : PASSED"
)

print(
    "Structured output           : PASSED"
)

print(
    "Research hypothesis status  : PASSED"
)

print("=" * 100)

AIR-LLM — NOTEBOOK 05.9
GENERATING STRUCTURED LLM-GUIDED STRATEGY RECOMMENDATIONS

----------------------------------------------------------------------------------------------------
LOADING NOTEBOOK 05.5 EVALUATION CONTEXT
----------------------------------------------------------------------------------------------------
Source file                 : /content/drive/MyDrive/AIR_LLM_Research/data/notebook_05/contexts/evaluation_context.csv
Rows loaded                 : 11
Columns loaded              : 19

----------------------------------------------------------------------------------------------------
EVALUATION POPULATION
----------------------------------------------------------------------------------------------------
Evaluation contexts          : 11
Expected from Notebook 05.5  : 11

----------------------------------------------------------------------------------------------------
LOADING CANDIDATE STRATEGY REGISTRY
----------------------------------------------------------

,dataset_id,contexts,recommendations
0,adult_income,3,15
1,diabetes_130us,8,40



----------------------------------------------------------------------------------------------------
TOP RECOMMENDATIONS
----------------------------------------------------------------------------------------------------


,dataset_id,feature,feature_type,missing_rate,rank,strategy_id,prior_score,confidence,estimated_cost
0,adult_income,native_country,categorical,0.017857,1,gradient_boosting,5.00,0.90,medium
1,adult_income,native_country,categorical,0.017857,2,random_forest,5.00,0.82,medium
2,adult_income,native_country,categorical,0.017857,3,random_sample,4.75,0.74,low
3,adult_income,native_country,categorical,0.017857,4,missforest,4.45,0.66,medium
4,adult_income,native_country,categorical,0.017857,5,mode,4.00,0.58,low
5,adult_income,occupation,categorical,0.056204,1,gradient_boosting,4.50,0.90,medium
6,adult_income,occupation,categorical,0.056204,2,random_forest,4.50,0.82,medium
7,adult_income,occupation,categorical,0.056204,3,random_sample,4.25,0.74,low
8,adult_income,occupation,categorical,0.056204,4,mode,4.00,0.66,low
9,adult_income,occupation,categorical,0.056204,5,mice,3.95,0.58,medium



NOTEBOOK 05.9 RECOMMENDATION STAGE COMPLETE
Evaluation population       : PASSED
Candidate compatibility     : PASSED
Ranking validation          : PASSED
Audit-only exclusion        : PASSED
Target-leakage protection   : PASSED
Recommendation count        : PASSED
Structured output           : PASSED
Research hypothesis status  : PASSED


In [12]:
# ============================================================
# NOTEBOOK 05.10 — RECOMMENDATION INPUT VALIDATION
# ============================================================

print("=" * 100)
print("AIR-LLM — NOTEBOOK 05.10")
print("RECOMMENDATION INPUT VALIDATION")
print("=" * 100)

# ------------------------------------------------------------
# Configuration
# ------------------------------------------------------------

REQUIRED_RECOMMENDATION_COLUMNS = [
    "dataset_id",
    "target",
    "feature",
    "feature_type",
    "missing_rate",
    "rank",
    "strategy_id",
    "family",
    "rationale",
    "strengths",
    "limitations",
    "applicability",
    "confidence",
    "estimated_cost",
    "recommendation_source",
    "recommendation_version"
]

ALLOWED_FEATURE_TYPES = {
    "numeric",
    "categorical"
}

ALLOWED_APPLICABILITY = {
    "high",
    "medium",
    "low"
}

ALLOWED_COSTS = {
    "low",
    "medium",
    "high"
}

# ------------------------------------------------------------
# Dependency validation
# ------------------------------------------------------------

required_objects = [
    "RECOMMENDATION_DF",
    "TOP_K"
]

missing_objects = [
    name
    for name in required_objects
    if name not in globals()
]

if missing_objects:
    raise RuntimeError(
        "Required objects are missing:\n"
        + "\n".join(missing_objects)
    )

if not isinstance(RECOMMENDATION_DF, pd.DataFrame):
    raise RuntimeError(
        "RECOMMENDATION_DF must be a pandas DataFrame."
    )

if RECOMMENDATION_DF.empty:
    raise RuntimeError(
        "RECOMMENDATION_DF is empty."
    )

if not isinstance(TOP_K, int) or TOP_K < 1:
    raise RuntimeError(
        f"Invalid TOP_K value: {TOP_K}"
    )

# ------------------------------------------------------------
# Resolve evaluation context
# ------------------------------------------------------------

if "evaluation_context" in globals():

    VALIDATION_CONTEXT_DF = evaluation_context.copy()

elif "EVALUATION_CONTEXT_DF" in globals():

    VALIDATION_CONTEXT_DF = EVALUATION_CONTEXT_DF.copy()

elif "CONTEXT_DF" in globals():

    if "recommendation_eligibility" in CONTEXT_DF.columns:

        VALIDATION_CONTEXT_DF = CONTEXT_DF[
            CONTEXT_DF[
                "recommendation_eligibility"
            ]
            .astype(str)
            .str.lower()
            == "eligible"
        ].copy()

    else:

        VALIDATION_CONTEXT_DF = CONTEXT_DF.copy()

else:

    raise RuntimeError(
        "No evaluation context dataframe is available."
    )

if VALIDATION_CONTEXT_DF.empty:
    raise RuntimeError(
        "Evaluation context is empty."
    )

# ------------------------------------------------------------
# Validate evaluation context schema
# ------------------------------------------------------------

required_context_columns = [
    "dataset_id",
    "feature"
]

missing_context_columns = [
    column
    for column in required_context_columns
    if column not in VALIDATION_CONTEXT_DF.columns
]

if missing_context_columns:
    raise RuntimeError(
        "Evaluation context is missing required columns:\n"
        + "\n".join(missing_context_columns)
    )

# ------------------------------------------------------------
# Normalize evaluation context
# ------------------------------------------------------------

VALIDATION_CONTEXT_DF["dataset_id"] = (
    VALIDATION_CONTEXT_DF["dataset_id"]
    .astype(str)
    .str.strip()
)

VALIDATION_CONTEXT_DF["feature"] = (
    VALIDATION_CONTEXT_DF["feature"]
    .astype(str)
    .str.strip()
)

# ------------------------------------------------------------
# Build expected recommendation targets
# ------------------------------------------------------------

EXPECTED_RECOMMENDATION_KEYS = (
    VALIDATION_CONTEXT_DF[
        [
            "dataset_id",
            "feature"
        ]
    ]
    .drop_duplicates()
    .sort_values(
        [
            "dataset_id",
            "feature"
        ]
    )
    .reset_index(drop=True)
)

expected_context_keys = set(
    zip(
        EXPECTED_RECOMMENDATION_KEYS["dataset_id"],
        EXPECTED_RECOMMENDATION_KEYS["feature"]
    )
)

print("\nEXPECTED RECOMMENDATION TARGETS")
print("-" * 100)

display(
    EXPECTED_RECOMMENDATION_KEYS
)

print(
    f"\nExpected evaluation contexts: "
    f"{len(EXPECTED_RECOMMENDATION_KEYS)}"
)

# ------------------------------------------------------------
# Recommendation schema validation
# ------------------------------------------------------------

missing_columns = [
    column
    for column in REQUIRED_RECOMMENDATION_COLUMNS
    if column not in RECOMMENDATION_DF.columns
]

if missing_columns:
    raise RuntimeError(
        "Recommendation table is missing required columns:\n"
        + "\n".join(missing_columns)
    )

# ------------------------------------------------------------
# Normalize recommendation dataframe
# ------------------------------------------------------------

VALIDATION_DF = RECOMMENDATION_DF.copy()

for column in [
    "dataset_id",
    "feature",
    "strategy_id",
    "feature_type",
    "family"
]:

    VALIDATION_DF[column] = (
        VALIDATION_DF[column]
        .astype(str)
        .str.strip()
    )

VALIDATION_DF["feature_type"] = (
    VALIDATION_DF["feature_type"]
    .str.lower()
)

VALIDATION_DF["family"] = (
    VALIDATION_DF["family"]
    .str.lower()
)

# ------------------------------------------------------------
# Recommendation context keys
# ------------------------------------------------------------

recommendation_context_keys = set(
    zip(
        VALIDATION_DF["dataset_id"],
        VALIDATION_DF["feature"]
    )
)

# ------------------------------------------------------------
# Missing recommendation targets
# ------------------------------------------------------------

MISSING_RECOMMENDATION_KEYS = (
    expected_context_keys
    -
    recommendation_context_keys
)

print("\nMISSING RECOMMENDATION TARGETS")
print("-" * 100)

if MISSING_RECOMMENDATION_KEYS:

    MISSING_RECOMMENDATION_DF = (
        pd.DataFrame(
            sorted(
                MISSING_RECOMMENDATION_KEYS
            ),
            columns=[
                "dataset_id",
                "feature"
            ]
        )
    )

    display(
        MISSING_RECOMMENDATION_DF
    )

    print(
        f"\nMissing recommendation targets: "
        f"{len(MISSING_RECOMMENDATION_DF)}"
    )

else:

    MISSING_RECOMMENDATION_DF = pd.DataFrame(
        columns=[
            "dataset_id",
            "feature"
        ]
    )

    print(
        "Missing recommendation targets: 0"
    )

# ------------------------------------------------------------
# Unexpected recommendation targets
# ------------------------------------------------------------

UNEXPECTED_RECOMMENDATION_KEYS = (
    recommendation_context_keys
    -
    expected_context_keys
)

print("\nUNEXPECTED RECOMMENDATION TARGETS")
print("-" * 100)

if UNEXPECTED_RECOMMENDATION_KEYS:

    UNEXPECTED_RECOMMENDATION_DF = (
        pd.DataFrame(
            sorted(
                UNEXPECTED_RECOMMENDATION_KEYS
            ),
            columns=[
                "dataset_id",
                "feature"
            ]
        )
    )

    display(
        UNEXPECTED_RECOMMENDATION_DF
    )

    print(
        f"\nUnexpected recommendation targets: "
        f"{len(UNEXPECTED_RECOMMENDATION_DF)}"
    )

else:

    UNEXPECTED_RECOMMENDATION_DF = pd.DataFrame(
        columns=[
            "dataset_id",
            "feature"
        ]
    )

    print(
        "Unexpected recommendation targets: 0"
    )

# ------------------------------------------------------------
# Recommendation coverage table
# ------------------------------------------------------------

RECOMMENDATION_COUNTS_DF = (
    VALIDATION_DF
    .groupby(
        [
            "dataset_id",
            "feature"
        ],
        as_index=False
    )
    .size()
    .rename(
        columns={
            "size": "recommendation_count"
        }
    )
)

RECOMMENDATION_COVERAGE_DF = (
    EXPECTED_RECOMMENDATION_KEYS
    .merge(
        RECOMMENDATION_COUNTS_DF,
        on=[
            "dataset_id",
            "feature"
        ],
        how="left"
    )
)

RECOMMENDATION_COVERAGE_DF[
    "recommendation_count"
] = (
    RECOMMENDATION_COVERAGE_DF[
        "recommendation_count"
    ]
    .fillna(0)
    .astype(int)
)

RECOMMENDATION_COVERAGE_DF[
    "expected_count"
] = TOP_K

RECOMMENDATION_COVERAGE_DF[
    "coverage_status"
] = np.where(
    RECOMMENDATION_COVERAGE_DF[
        "recommendation_count"
    ]
    ==
    TOP_K,
    "PASSED",
    "FAILED"
)

print("\nRECOMMENDATION COVERAGE")
print("-" * 100)

display(
    RECOMMENDATION_COVERAGE_DF
)

# ------------------------------------------------------------
# Fail early on context coverage
# ------------------------------------------------------------

coverage_failures = (
    RECOMMENDATION_COVERAGE_DF[
        RECOMMENDATION_COVERAGE_DF[
            "coverage_status"
        ]
        != "PASSED"
    ]
)

if not coverage_failures.empty:

    raise RuntimeError(
        "\nRecommendation coverage validation failed.\n\n"
        f"Expected evaluation contexts : "
        f"{len(EXPECTED_RECOMMENDATION_KEYS)}\n"
        f"TOP-K recommendations        : "
        f"{TOP_K}\n"
        f"Expected total rows           : "
        f"{len(EXPECTED_RECOMMENDATION_KEYS) * TOP_K}\n"
        f"Observed total rows           : "
        f"{len(VALIDATION_DF)}\n\n"
        "Contexts with incorrect recommendation counts:\n"
        + str(
            coverage_failures
        )
        + "\n\nMissing recommendation targets:\n"
        + str(
            sorted(
                MISSING_RECOMMENDATION_KEYS
            )
        )
    )

# ------------------------------------------------------------
# Candidate strategy registry
# ------------------------------------------------------------

if "CANDIDATE_STRATEGY_DF" not in globals():
    raise RuntimeError(
        "CANDIDATE_STRATEGY_DF is not available."
    )

required_strategy_columns = [
    "strategy_id",
    "family",
    "feature_types"
]

missing_strategy_columns = [
    column
    for column in required_strategy_columns
    if column not in CANDIDATE_STRATEGY_DF.columns
]

if missing_strategy_columns:
    raise RuntimeError(
        "CANDIDATE_STRATEGY_DF is missing required columns:\n"
        + "\n".join(missing_strategy_columns)
    )

STRATEGY_LOOKUP = {
    str(row["strategy_id"]): row
    for row in CANDIDATE_STRATEGY_DF.to_dict(
        orient="records"
    )
}

allowed_strategy_ids = set(
    CANDIDATE_STRATEGY_DF[
        "strategy_id"
    ]
    .astype(str)
)

# ------------------------------------------------------------
# Row-level validation
# ------------------------------------------------------------

validation_rows = []

for _, row in VALIDATION_DF.iterrows():

    strategy_id = row["strategy_id"]

    valid_strategy = (
        strategy_id
        in allowed_strategy_ids
    )

    try:

        rank_value = int(row["rank"])

        valid_rank = (
            1
            <= rank_value
            <= TOP_K
        )

    except Exception:

        valid_rank = False

    confidence = safe_float(
        row["confidence"],
        default=np.nan
    )

    valid_confidence = (
        np.isfinite(confidence)
        and
        0.0
        <= confidence
        <= 1.0
    )

    valid_feature_type = (
        row["feature_type"]
        in ALLOWED_FEATURE_TYPES
    )

    valid_applicability = (
        str(
            row["applicability"]
        )
        .strip()
        .lower()
        in ALLOWED_APPLICABILITY
    )

    valid_cost = (
        str(
            row["estimated_cost"]
        )
        .strip()
        .lower()
        in ALLOWED_COSTS
    )

    if valid_strategy:

        strategy = STRATEGY_LOOKUP[
            strategy_id
        ]

        strategy_feature_types = [
            str(x).lower()
            for x in strategy[
                "feature_types"
            ]
        ]

        type_compatible = (
            row["feature_type"]
            in strategy_feature_types
        )

        family_consistent = (
            str(
                strategy["family"]
            )
            .strip()
            .lower()
            ==
            row["family"]
        )

    else:

        type_compatible = False
        family_consistent = False

    context_key = (
        row["dataset_id"],
        row["feature"]
    )

    context_exists = (
        context_key
        in expected_context_keys
    )

    required_text_fields = [
        row["rationale"],
        row["strengths"],
        row["limitations"]
    ]

    text_complete = all(
        pd.notna(value)
        and
        str(value).strip() != ""
        for value in required_text_fields
    )

    valid = all([
        valid_strategy,
        valid_rank,
        valid_confidence,
        valid_feature_type,
        valid_applicability,
        valid_cost,
        type_compatible,
        family_consistent,
        context_exists,
        text_complete
    ])

    validation_rows.append({

        "dataset_id":
            row["dataset_id"],

        "feature":
            row["feature"],

        "strategy_id":
            strategy_id,

        "valid_strategy":
            valid_strategy,

        "valid_rank":
            valid_rank,

        "valid_confidence":
            valid_confidence,

        "valid_feature_type":
            valid_feature_type,

        "valid_applicability":
            valid_applicability,

        "valid_cost":
            valid_cost,

        "type_compatible":
            type_compatible,

        "family_consistent":
            family_consistent,

        "context_exists":
            context_exists,

        "text_complete":
            text_complete,

        "valid":
            valid
    })

RECOMMENDATION_VALIDATION_DF = pd.DataFrame(
    validation_rows
)

# ------------------------------------------------------------
# Duplicate recommendation validation
# ------------------------------------------------------------

duplicate_mask = (
    VALIDATION_DF[
        [
            "dataset_id",
            "feature",
            "strategy_id"
        ]
    ]
    .duplicated(
        keep=False
    )
)

if duplicate_mask.any():

    duplicates = (
        VALIDATION_DF.loc[
            duplicate_mask,
            [
                "dataset_id",
                "feature",
                "strategy_id"
            ]
        ]
        .drop_duplicates()
        .to_dict(
            orient="records"
        )
    )

    raise RuntimeError(
        "Duplicate strategy recommendations detected:\n"
        + str(duplicates)
    )

# ------------------------------------------------------------
# Rank continuity validation
# ------------------------------------------------------------

rank_errors = []

for (
    dataset_id,
    feature
), group in VALIDATION_DF.groupby(
    [
        "dataset_id",
        "feature"
    ]
):

    try:

        ranks = sorted(
            group["rank"]
            .astype(int)
            .tolist()
        )

    except Exception:

        rank_errors.append({
            "dataset_id":
                dataset_id,
            "feature":
                feature,
            "error":
                "Non-integer rank detected"
        })

        continue

    expected_ranks = list(
        range(
            1,
            TOP_K + 1
        )
    )

    if ranks != expected_ranks:

        rank_errors.append({
            "dataset_id":
                dataset_id,
            "feature":
                feature,
            "observed_ranks":
                ranks,
            "expected_ranks":
                expected_ranks
        })

if rank_errors:

    raise RuntimeError(
        "Rank continuity validation failed:\n"
        + str(rank_errors)
    )

# ------------------------------------------------------------
# Audit-only exclusion
# ------------------------------------------------------------

if "recommendation_eligibility" in VALIDATION_CONTEXT_DF.columns:

    audit_mask = (
        VALIDATION_CONTEXT_DF[
            "recommendation_eligibility"
        ]
        .astype(str)
        .str.lower()
        == "audit_only"
    )

    audit_context_keys = set(
        zip(
            VALIDATION_CONTEXT_DF.loc[
                audit_mask,
                "dataset_id"
            ].astype(str),

            VALIDATION_CONTEXT_DF.loc[
                audit_mask,
                "feature"
            ].astype(str)
        )
    )

else:

    audit_context_keys = set()

audit_leakage = (
    recommendation_context_keys
    &
    audit_context_keys
)

if audit_leakage:

    raise RuntimeError(
        "Audit-only contexts entered recommendation evaluation:\n"
        + str(
            sorted(
                audit_leakage
            )
        )
    )

# ------------------------------------------------------------
# Recommendation source validation
# ------------------------------------------------------------

sources = (
    VALIDATION_DF[
        "recommendation_source"
    ]
    .dropna()
    .astype(str)
    .str.strip()
    .unique()
)

if len(sources) != 1:

    raise RuntimeError(
        "Recommendation source is inconsistent:\n"
        + str(sources)
    )

versions = (
    VALIDATION_DF[
        "recommendation_version"
    ]
    .dropna()
    .astype(str)
    .str.strip()
    .unique()
)

if len(versions) != 1:

    raise RuntimeError(
        "Recommendation version is inconsistent:\n"
        + str(versions)
    )

# ------------------------------------------------------------
# Expected recommendation count
# ------------------------------------------------------------

expected_recommendations = (
    len(expected_context_keys)
    *
    TOP_K
)

actual_recommendations = (
    len(VALIDATION_DF)
)

if actual_recommendations != expected_recommendations:

    raise RuntimeError(
        "Recommendation count mismatch.\n"
        f"Expected: {expected_recommendations}\n"
        f"Observed: {actual_recommendations}"
    )

# ------------------------------------------------------------
# Save validation report
# ------------------------------------------------------------

VALIDATION_PATH = (
    VALIDATION_DIR
    /
    "recommendation_validation.csv"
)

RECOMMENDATION_VALIDATION_DF.to_csv(
    VALIDATION_PATH,
    index=False
)

COVERAGE_PATH = (
    VALIDATION_DIR
    /
    "recommendation_coverage.csv"
)

RECOMMENDATION_COVERAGE_DF.to_csv(
    COVERAGE_PATH,
    index=False
)

# ------------------------------------------------------------
# Summary statistics
# ------------------------------------------------------------

dataset_summary = (
    VALIDATION_DF
    .groupby("dataset_id")
    .agg(
        contexts=(
            "feature",
            "nunique"
        ),
        recommendations=(
            "strategy_id",
            "count"
        )
    )
    .reset_index()
)

# ------------------------------------------------------------
# Final structured validation
# ------------------------------------------------------------

if not RECOMMENDATION_VALIDATION_DF[
    "valid"
].all():

    invalid_rows = (
        RECOMMENDATION_VALIDATION_DF[
            ~RECOMMENDATION_VALIDATION_DF[
                "valid"
            ]
        ]
    )

    raise RuntimeError(
        "Recommendation validation failed.\n\n"
        + str(
            invalid_rows
        )
    )

# ------------------------------------------------------------
# Final report
# ------------------------------------------------------------

print("\n" + "-" * 100)
print("VALIDATION RESULTS")
print("-" * 100)

print(
    "Required columns              : PASSED"
)

print(
    f"Recommendation rows            : "
    f"{actual_recommendations}"
)

print(
    f"Expected recommendation rows   : "
    f"{expected_recommendations}"
)

print(
    f"Evaluation contexts             : "
    f"{len(expected_context_keys)}"
)

print(
    f"TOP-K per context               : "
    f"{TOP_K}"
)

print(
    "Recommendation coverage        : PASSED"
)

print(
    "Strategy registry validation    : PASSED"
)

print(
    "Feature-type compatibility      : PASSED"
)

print(
    "Strategy-family consistency     : PASSED"
)

print(
    "Confidence validation           : PASSED"
)

print(
    "Applicability validation        : PASSED"
)

print(
    "Computational-cost validation   : PASSED"
)

print(
    "Context coverage                : PASSED"
)

print(
    "Duplicate strategy check        : PASSED"
)

print(
    "Rank continuity                 : PASSED"
)

print(
    "Audit-only exclusion            : PASSED"
)

print(
    "Recommendation source           : PASSED"
)

print(
    "Recommendation version          : PASSED"
)

print(
    "Recommendation count            : PASSED"
)

print(
    "Structured recommendation       : PASSED"
)

print("\n" + "-" * 100)
print("RECOMMENDATION POPULATION")
print("-" * 100)

display(
    dataset_summary
)

print("\n" + "=" * 100)
print("NOTEBOOK 05.10 RECOMMENDATION VALIDATION COMPLETE")
print("=" * 100)

print(
    f"Validated recommendation rows : "
    f"{actual_recommendations}"
)

print(
    f"Validated contexts             : "
    f"{len(expected_context_keys)}"
)

print(
    f"Expected TOP-K recommendations : "
    f"{TOP_K}"
)

print(
    f"Validation report              : "
    f"{VALIDATION_PATH}"
)

print(
    f"Coverage report                : "
    f"{COVERAGE_PATH}"
)

print(
    "STATUS: PASSED"
)

print("=" * 100)

AIR-LLM — NOTEBOOK 05.10
RECOMMENDATION INPUT VALIDATION

EXPECTED RECOMMENDATION TARGETS
----------------------------------------------------------------------------------------------------


,dataset_id,feature
0,adult_income,native_country
1,adult_income,occupation
2,adult_income,workclass
3,diabetes_130us,diag_1
4,diabetes_130us,diag_2
5,diabetes_130us,diag_3
6,diabetes_130us,max_glu_serum
7,diabetes_130us,medical_specialty
8,diabetes_130us,payer_code
9,diabetes_130us,race



Expected evaluation contexts: 11

MISSING RECOMMENDATION TARGETS
----------------------------------------------------------------------------------------------------
Missing recommendation targets: 0

UNEXPECTED RECOMMENDATION TARGETS
----------------------------------------------------------------------------------------------------
Unexpected recommendation targets: 0

RECOMMENDATION COVERAGE
----------------------------------------------------------------------------------------------------


,dataset_id,feature,recommendation_count,expected_count,coverage_status
0,adult_income,native_country,5,5,PASSED
1,adult_income,occupation,5,5,PASSED
2,adult_income,workclass,5,5,PASSED
3,diabetes_130us,diag_1,5,5,PASSED
4,diabetes_130us,diag_2,5,5,PASSED
5,diabetes_130us,diag_3,5,5,PASSED
6,diabetes_130us,max_glu_serum,5,5,PASSED
7,diabetes_130us,medical_specialty,5,5,PASSED
8,diabetes_130us,payer_code,5,5,PASSED
9,diabetes_130us,race,5,5,PASSED



----------------------------------------------------------------------------------------------------
VALIDATION RESULTS
----------------------------------------------------------------------------------------------------
Required columns              : PASSED
Recommendation rows            : 55
Expected recommendation rows   : 55
Evaluation contexts             : 11
TOP-K per context               : 5
Recommendation coverage        : PASSED
Strategy registry validation    : PASSED
Feature-type compatibility      : PASSED
Strategy-family consistency     : PASSED
Confidence validation           : PASSED
Applicability validation        : PASSED
Computational-cost validation   : PASSED
Context coverage                : PASSED
Duplicate strategy check        : PASSED
Rank continuity                 : PASSED
Audit-only exclusion            : PASSED
Recommendation source           : PASSED
Recommendation version          : PASSED
Recommendation count            : PASSED
Structured recommenda

,dataset_id,contexts,recommendations
0,adult_income,3,15
1,diabetes_130us,8,40



NOTEBOOK 05.10 RECOMMENDATION VALIDATION COMPLETE
Validated recommendation rows : 55
Validated contexts             : 11
Expected TOP-K recommendations : 5
Validation report              : /content/drive/MyDrive/AIR_LLM_Research/data/notebook_05/validation/recommendation_validation.csv
Coverage report                : /content/drive/MyDrive/AIR_LLM_Research/data/notebook_05/validation/recommendation_coverage.csv
STATUS: PASSED


In [13]:
# ============================================================
# NOTEBOOK 05.11 — RECOMMENDATION COVERAGE
# ============================================================

print("=" * 100)
print("AIR-LLM — NOTEBOOK 05.11")
print("RECOMMENDATION COVERAGE ANALYSIS")
print("=" * 100)

# ------------------------------------------------------------
# 1. Resolve evaluation-eligible context
# ------------------------------------------------------------

if "evaluation_context" in globals():

    COVERAGE_CONTEXT_DF = evaluation_context.copy()

elif "EVALUATION_CONTEXT_DF" in globals():

    COVERAGE_CONTEXT_DF = EVALUATION_CONTEXT_DF.copy()

else:

    raise RuntimeError(
        "No evaluation-eligible context dataframe is available."
    )

if COVERAGE_CONTEXT_DF.empty:

    raise RuntimeError(
        "Evaluation-eligible context is empty."
    )

# ------------------------------------------------------------
# 2. Normalize context keys
# ------------------------------------------------------------

required_context_columns = [
    "dataset_id",
    "target",
    "feature"
]

missing_context_columns = [
    column
    for column in required_context_columns
    if column not in COVERAGE_CONTEXT_DF.columns
]

if missing_context_columns:

    raise RuntimeError(
        "Evaluation context is missing required columns:\n"
        + "\n".join(missing_context_columns)
    )

COVERAGE_CONTEXT_DF = (
    COVERAGE_CONTEXT_DF[
        required_context_columns
    ]
    .copy()
)

COVERAGE_CONTEXT_DF["dataset_id"] = (
    COVERAGE_CONTEXT_DF["dataset_id"]
    .astype(str)
    .str.strip()
)

COVERAGE_CONTEXT_DF["feature"] = (
    COVERAGE_CONTEXT_DF["feature"]
    .astype(str)
    .str.strip()
)

COVERAGE_CONTEXT_DF["target"] = (
    COVERAGE_CONTEXT_DF["target"]
    .astype(str)
    .str.strip()
)

# ------------------------------------------------------------
# 3. Explicit audit-only protection
# ------------------------------------------------------------
#
# These contexts must never enter recommendation coverage.
#
# This protection is intentionally retained even though
# evaluation_context should already contain only eligible
# contexts.
# ------------------------------------------------------------

AUDIT_ONLY_KEYS = {
    (
        "diabetes_130us",
        "a1cresult"
    )
}

COVERAGE_CONTEXT_DF = (
    COVERAGE_CONTEXT_DF[
        ~COVERAGE_CONTEXT_DF.apply(
            lambda row:
            (
                row["dataset_id"],
                row["feature"]
            ) in AUDIT_ONLY_KEYS,
            axis=1
        )
    ]
    .drop_duplicates(
        subset=[
            "dataset_id",
            "feature"
        ]
    )
    .reset_index(drop=True)
)

# ------------------------------------------------------------
# 4. Expected evaluation population
# ------------------------------------------------------------

EXPECTED_FEATURES = (
    COVERAGE_CONTEXT_DF[
        [
            "dataset_id",
            "target",
            "feature"
        ]
    ]
    .drop_duplicates()
    .reset_index(drop=True)
)

if EXPECTED_FEATURES.empty:

    raise RuntimeError(
        "No evaluation-eligible incomplete features were found."
    )

# ------------------------------------------------------------
# 5. Resolve recommendation dataframe
# ------------------------------------------------------------

if "RECOMMENDATION_DF" not in globals():

    raise RuntimeError(
        "RECOMMENDATION_DF is not available."
    )

if RECOMMENDATION_DF.empty:

    raise RuntimeError(
        "RECOMMENDATION_DF is empty."
    )

RECOMMENDATION_DF = RECOMMENDATION_DF.copy()

RECOMMENDATION_DF["dataset_id"] = (
    RECOMMENDATION_DF["dataset_id"]
    .astype(str)
    .str.strip()
)

RECOMMENDATION_DF["feature"] = (
    RECOMMENDATION_DF["feature"]
    .astype(str)
    .str.strip()
)

# ------------------------------------------------------------
# 6. Explicit audit-only leakage protection
# ------------------------------------------------------------

recommendation_audit_keys = set(
    zip(
        RECOMMENDATION_DF["dataset_id"],
        RECOMMENDATION_DF["feature"]
    )
) & AUDIT_ONLY_KEYS

if recommendation_audit_keys:

    raise RuntimeError(
        "Audit-only contexts detected in recommendations:\n"
        + str(
            sorted(
                recommendation_audit_keys
            )
        )
    )

# ------------------------------------------------------------
# 7. Recommendation coverage
# ------------------------------------------------------------

RECOMMENDED_FEATURES = (
    RECOMMENDATION_DF[
        [
            "dataset_id",
            "feature"
        ]
    ]
    .drop_duplicates()
)

coverage = (
    EXPECTED_FEATURES
    .merge(
        RECOMMENDED_FEATURES,
        on=[
            "dataset_id",
            "feature"
        ],
        how="left",
        indicator=True
    )
)

coverage["recommended"] = (
    coverage["_merge"]
    == "both"
)

# ------------------------------------------------------------
# 8. Recommendation count per feature
# ------------------------------------------------------------

recommendation_counts = (
    RECOMMENDATION_DF
    .groupby(
        [
            "dataset_id",
            "feature"
        ]
    )
    .size()
    .rename(
        "recommendation_count"
    )
    .reset_index()
)

RECOMMENDATION_COVERAGE_DF = (
    EXPECTED_FEATURES
    .merge(
        coverage[
            [
                "dataset_id",
                "feature",
                "recommended"
            ]
        ],
        on=[
            "dataset_id",
            "feature"
        ],
        how="left"
    )
    .merge(
        recommendation_counts,
        on=[
            "dataset_id",
            "feature"
        ],
        how="left"
    )
)

RECOMMENDATION_COVERAGE_DF[
    "recommendation_count"
] = (
    RECOMMENDATION_COVERAGE_DF[
        "recommendation_count"
    ]
    .fillna(0)
    .astype(int)
)

RECOMMENDATION_COVERAGE_DF[
    "recommended"
] = (
    RECOMMENDATION_COVERAGE_DF[
        "recommended"
    ]
    .fillna(False)
    .astype(bool)
)

# ------------------------------------------------------------
# 9. Validate recommendation counts
# ------------------------------------------------------------

invalid_counts = (
    RECOMMENDATION_COVERAGE_DF[
        RECOMMENDATION_COVERAGE_DF[
            "recommendation_count"
        ] != TOP_K
    ]
)

if not invalid_counts.empty:

    raise RuntimeError(
        "One or more evaluation contexts do not contain "
        f"exactly TOP_K={TOP_K} recommendations:\n"
        + str(
            invalid_counts[
                [
                    "dataset_id",
                    "feature",
                    "recommendation_count"
                ]
            ]
            .to_dict(
                orient="records"
            )
        )
    )

# ------------------------------------------------------------
# 10. Dataset-level coverage summary
# ------------------------------------------------------------

coverage_summary_rows = []

for dataset_id in DATASETS:

    expected = (
        EXPECTED_FEATURES[
            EXPECTED_FEATURES[
                "dataset_id"
            ]
            == str(dataset_id)
        ]
    )

    expected_count = len(
        expected
    )

    recommended_count = int(
        RECOMMENDATION_COVERAGE_DF.loc[
            RECOMMENDATION_COVERAGE_DF[
                "dataset_id"
            ]
            == str(dataset_id),
            "recommended"
        ].sum()
    )

    if expected_count > 0:

        coverage_rate = (
            recommended_count
            /
            expected_count
        )

        coverage_status = (
            "COMPLETE"
            if recommended_count
            == expected_count
            else
            "INCOMPLETE"
        )

    else:

        coverage_rate = np.nan
        coverage_status = "NO_ELIGIBLE_FEATURES"

    target = ""

    if "TARGET_REGISTRY" in globals():

        target = TARGET_REGISTRY.get(
            dataset_id,
            ""
        )

    elif not expected.empty:

        target = expected.iloc[0]["target"]

    coverage_summary_rows.append({

        "dataset_id":
            dataset_id,

        "target":
            target,

        "expected_features":
            expected_count,

        "recommended_features":
            recommended_count,

        "coverage_rate":
            coverage_rate,

        "coverage_status":
            coverage_status
    })

COVERAGE_SUMMARY_DF = pd.DataFrame(
    coverage_summary_rows
)

# ------------------------------------------------------------
# 11. Display dataset-level coverage
# ------------------------------------------------------------

print("\nCOVERAGE SUMMARY")
print("-" * 100)

display(
    COVERAGE_SUMMARY_DF
)

# ------------------------------------------------------------
# 12. Overall evaluation coverage
# ------------------------------------------------------------

overall_expected = len(
    EXPECTED_FEATURES
)

overall_recommended = int(
    RECOMMENDATION_COVERAGE_DF[
        "recommended"
    ].sum()
)

if overall_expected > 0:

    overall_coverage = (
        overall_recommended
        /
        overall_expected
    )

else:

    overall_coverage = np.nan

print("\nOVERALL COVERAGE")
print("-" * 100)

print(
    f"Evaluation-eligible incomplete features : "
    f"{overall_expected}"
)

print(
    f"Recommended features                    : "
    f"{overall_recommended}"
)

print(
    f"Overall coverage                        : "
    f"{overall_coverage:.4f}"
    if np.isfinite(overall_coverage)
    else
    "Overall coverage                        : N/A"
)

# ------------------------------------------------------------
# 13. Identify uncovered eligible features
# ------------------------------------------------------------

UNCOVERED_FEATURE_DF = (
    RECOMMENDATION_COVERAGE_DF[
        ~RECOMMENDATION_COVERAGE_DF[
            "recommended"
        ]
    ]
    [
        [
            "dataset_id",
            "target",
            "feature",
            "recommendation_count"
        ]
    ]
)

# ------------------------------------------------------------
# 14. Coverage validation
# ------------------------------------------------------------

if not UNCOVERED_FEATURE_DF.empty:

    print(
        "\nUNCOVERED EVALUATION FEATURES"
    )

    print(
        "-" * 100
    )

    display(
        UNCOVERED_FEATURE_DF
    )

    raise RuntimeError(
        "Recommendation coverage is incomplete "
        "for evaluation-eligible features."
    )

# ------------------------------------------------------------
# 15. Audit-only confirmation
# ------------------------------------------------------------

audit_present_in_expected = (
    EXPECTED_FEATURES.apply(
        lambda row:
        (
            row["dataset_id"],
            row["feature"]
        ) in AUDIT_ONLY_KEYS,
        axis=1
    )
    .any()
)

if audit_present_in_expected:

    raise RuntimeError(
        "Audit-only context entered the expected "
        "recommendation coverage population."
    )

# ------------------------------------------------------------
# 16. Final status
# ------------------------------------------------------------

print(
    "\nRecommendation coverage : PASSED"
)

print(
    "Evaluation population    : 11"
)

print(
    "Recommended contexts     : 11"
)

print(
    "Overall coverage         : 100%"
)

print(
    "Audit-only contexts     : 0"
)

print(
    "Uncovered eligible       : 0"
)

print("=" * 100)
print(
    "NOTEBOOK 05.11 RECOMMENDATION COVERAGE COMPLETE"
)
print("=" * 100)

AIR-LLM — NOTEBOOK 05.11
RECOMMENDATION COVERAGE ANALYSIS

COVERAGE SUMMARY
----------------------------------------------------------------------------------------------------


,dataset_id,target,expected_features,recommended_features,coverage_rate,coverage_status
0,adult_income,income,3,3,1.0,COMPLETE
1,bank_marketing,y,0,0,NaN,NO_ELIGIBLE_FEATURES
2,diabetes_130us,readmitted,8,8,1.0,COMPLETE



OVERALL COVERAGE
----------------------------------------------------------------------------------------------------
Evaluation-eligible incomplete features : 11
Recommended features                    : 11
Overall coverage                        : 1.0000

Recommendation coverage : PASSED
Evaluation population    : 11
Recommended contexts     : 11
Overall coverage         : 100%
Audit-only contexts     : 0
Uncovered eligible       : 0
NOTEBOOK 05.11 RECOMMENDATION COVERAGE COMPLETE


In [14]:
# ============================================================
# NOTEBOOK 05.12 — RANKING AND CONFIDENCE ANALYSIS
# ============================================================

print("=" * 100)
print("AIR-LLM — NOTEBOOK 05.12")
print("RANKING AND CONFIDENCE ANALYSIS")
print("=" * 100)


# ------------------------------------------------------------
# Ranking uniqueness
# ------------------------------------------------------------

ranking_counts = (
    RECOMMENDATION_DF
    .groupby(
        [
            "dataset_id",
            "feature"
        ]
    )["rank"]
    .nunique()
)


if not (
    ranking_counts
    == TOP_K
).all():

    raise RuntimeError(
        "Ranking uniqueness/top-K validation failed."
    )


# ------------------------------------------------------------
# Confidence validation
# ------------------------------------------------------------

if not (
    RECOMMENDATION_DF[
        "confidence"
    ]
    .between(
        0,
        1
    )
).all():

    raise RuntimeError(
        "Confidence values outside [0,1]."
    )


# ------------------------------------------------------------
# Top recommendation table
# ------------------------------------------------------------

TOP_RECOMMENDATIONS_DF = (
    RECOMMENDATION_DF[
        RECOMMENDATION_DF[
            "rank"
        ]
        == 1
    ]
    .copy()
    .sort_values(
        [
            "dataset_id",
            "feature"
        ]
    )
)


TOP_RECOMMENDATIONS_PATH = (
    RECOMMENDATION_DIR
    / "top_recommendations.csv"
)


TOP_RECOMMENDATIONS_DF.to_csv(
    TOP_RECOMMENDATIONS_PATH,
    index=False
)


print(
    "Ranking uniqueness : PASSED"
)

print(
    "TOP-K constraint   : PASSED"
)

print(
    "Confidence range   : PASSED"
)


print("\nTOP RECOMMENDATIONS")
print("-" * 100)

display(
    TOP_RECOMMENDATIONS_DF[
        [
            "dataset_id",
            "feature",
            "feature_type",
            "strategy_id",
            "confidence",
            "estimated_cost"
        ]
    ]
)

AIR-LLM — NOTEBOOK 05.12
RANKING AND CONFIDENCE ANALYSIS
Ranking uniqueness : PASSED
TOP-K constraint   : PASSED
Confidence range   : PASSED

TOP RECOMMENDATIONS
----------------------------------------------------------------------------------------------------


,dataset_id,feature,feature_type,strategy_id,confidence,estimated_cost
0,adult_income,native_country,categorical,gradient_boosting,0.9,medium
5,adult_income,occupation,categorical,gradient_boosting,0.9,medium
10,adult_income,workclass,categorical,gradient_boosting,0.9,medium
15,diabetes_130us,diag_1,categorical,gradient_boosting,0.9,medium
20,diabetes_130us,diag_2,categorical,gradient_boosting,0.9,medium
25,diabetes_130us,diag_3,categorical,gradient_boosting,0.9,medium
30,diabetes_130us,max_glu_serum,categorical,gradient_boosting,0.9,medium
35,diabetes_130us,medical_specialty,categorical,gradient_boosting,0.9,medium
40,diabetes_130us,payer_code,categorical,gradient_boosting,0.9,medium
45,diabetes_130us,race,categorical,gradient_boosting,0.9,medium


In [16]:
# ============================================================
# NOTEBOOK 05.13 — PERSISTENCE
# ============================================================

print("=" * 100)
print("AIR-LLM — NOTEBOOK 05.13")
print("PERSISTING NOTEBOOK 05 OUTPUTS")
print("=" * 100)

from pathlib import Path
import json
from datetime import datetime, timezone

# ------------------------------------------------------------
# Canonical AIR-LLM research project root
# ------------------------------------------------------------

PROJECT_ROOT = Path(
    "/content/drive/MyDrive/AIR_LLM_Research"
)

if not PROJECT_ROOT.exists():
    raise RuntimeError(
        f"Research project root does not exist: {PROJECT_ROOT}"
    )

# ------------------------------------------------------------
# Notebook 05 directories
# ------------------------------------------------------------

NOTEBOOK_05_DIR = (
    PROJECT_ROOT
    / "data"
    / "notebook_05"
)

CONTEXT_DIR = (
    NOTEBOOK_05_DIR
    / "contexts"
)

RECOMMENDATION_DIR = (
    NOTEBOOK_05_DIR
    / "recommendations"
)

METADATA_DIR = (
    NOTEBOOK_05_DIR
    / "metadata"
)

VALIDATION_DIR = (
    NOTEBOOK_05_DIR
    / "validation"
)

# ------------------------------------------------------------
# Create directories
# ------------------------------------------------------------

for directory in [
    NOTEBOOK_05_DIR,
    CONTEXT_DIR,
    RECOMMENDATION_DIR,
    METADATA_DIR,
    VALIDATION_DIR
]:
    directory.mkdir(
        parents=True,
        exist_ok=True
    )

# ------------------------------------------------------------
# Required Notebook 05 artifacts
# ------------------------------------------------------------

REQUIRED_OBJECTS = {
    "CONTEXT_DF":
        "LLM context",

    "CANDIDATE_STRATEGY_DF":
        "Candidate strategy pool",

    "RECOMMENDATION_DF":
        "LLM recommendations",

    "TOP_RECOMMENDATIONS_DF":
        "Top recommendations",

    "RECOMMENDATION_VALIDATION_DF":
        "Recommendation validation",

    "RECOMMENDATION_COVERAGE_DF":
        "Recommendation coverage",

    "RECOMMENDATION_PROFILE_DF":
        "Recommendation profile"
}

missing_objects = [
    name
    for name in REQUIRED_OBJECTS
    if name not in globals()
]

if missing_objects:
    raise RuntimeError(
        "Required Notebook 05 objects are missing:\n"
        + "\n".join(
            f"- {name}: {REQUIRED_OBJECTS[name]}"
            for name in missing_objects
        )
    )

# ------------------------------------------------------------
# Output paths
# ------------------------------------------------------------

CONTEXT_PATH = (
    CONTEXT_DIR
    / "llm_context.csv"
)

RECOMMENDATIONS_PATH = (
    RECOMMENDATION_DIR
    / "llm_recommendations.csv"
)

TOP_RECOMMENDATIONS_PATH = (
    RECOMMENDATION_DIR
    / "top_recommendations.csv"
)

CANDIDATE_POOL_PATH = (
    METADATA_DIR
    / "candidate_strategy_pool.csv"
)

VALIDATION_PATH = (
    VALIDATION_DIR
    / "recommendation_validation.csv"
)

COVERAGE_PATH = (
    VALIDATION_DIR
    / "recommendation_coverage.csv"
)

PROFILE_PATH = (
    CONTEXT_DIR
    / "recommendation_profile.csv"
)

# ------------------------------------------------------------
# Persist LLM context
# ------------------------------------------------------------

CONTEXT_DF.to_csv(
    CONTEXT_PATH,
    index=False
)

# ------------------------------------------------------------
# Persist candidate strategy pool
# ------------------------------------------------------------

CANDIDATE_STRATEGY_DF.to_csv(
    CANDIDATE_POOL_PATH,
    index=False
)

# ------------------------------------------------------------
# Persist recommendations
# ------------------------------------------------------------

RECOMMENDATION_DF.to_csv(
    RECOMMENDATIONS_PATH,
    index=False
)

# ------------------------------------------------------------
# Persist top recommendations
# ------------------------------------------------------------

TOP_RECOMMENDATIONS_DF.to_csv(
    TOP_RECOMMENDATIONS_PATH,
    index=False
)

# ------------------------------------------------------------
# Persist validation
# ------------------------------------------------------------

RECOMMENDATION_VALIDATION_DF.to_csv(
    VALIDATION_PATH,
    index=False
)

# ------------------------------------------------------------
# Persist coverage
# ------------------------------------------------------------

RECOMMENDATION_COVERAGE_DF.to_csv(
    COVERAGE_PATH,
    index=False
)

# ------------------------------------------------------------
# Persist recommendation profile
# ------------------------------------------------------------

RECOMMENDATION_PROFILE_DF.to_csv(
    PROFILE_PATH,
    index=False
)

# ------------------------------------------------------------
# Verify persisted artifacts
# ------------------------------------------------------------

PERSISTED_ARTIFACTS = {
    "llm_context":
        CONTEXT_PATH,

    "llm_recommendations":
        RECOMMENDATIONS_PATH,

    "top_recommendations":
        TOP_RECOMMENDATIONS_PATH,

    "candidate_strategy_pool":
        CANDIDATE_POOL_PATH,

    "recommendation_validation":
        VALIDATION_PATH,

    "recommendation_coverage":
        COVERAGE_PATH,

    "recommendation_profile":
        PROFILE_PATH
}

persistence_errors = []

for name, path in PERSISTED_ARTIFACTS.items():

    if not path.exists():
        persistence_errors.append(
            f"{name}: file not created"
        )

    elif path.stat().st_size == 0:
        persistence_errors.append(
            f"{name}: file is empty"
        )

if persistence_errors:
    raise RuntimeError(
        "Notebook 05 persistence verification failed:\n"
        + "\n".join(persistence_errors)
    )

# ------------------------------------------------------------
# Manifest
# ------------------------------------------------------------

MANIFEST_PATH = (
    METADATA_DIR
    / "notebook_05_manifest.json"
)

manifest = {
    "notebook": "05",
    "name": "LLM Recommendation Engine",

    "project_root": str(
        PROJECT_ROOT
    ),

    "notebook_directory": str(
        NOTEBOOK_05_DIR
    ),

    "datasets": [
        "adult_income",
        "bank_marketing",
        "diabetes_130us"
    ],

    "profiled_features": 80,

    "incomplete_features": 12,

    "candidate_strategies": 13,

    "candidates_per_feature": 5,

    "evaluation_contexts": int(
        len(CONTEXT_DF)
    ),

    "recommendation_rows": int(
        len(RECOMMENDATION_DF)
    ),

    "validation_rows": int(
        len(RECOMMENDATION_VALIDATION_DF)
    ),

    "created_at_utc": (
        datetime.now(
            timezone.utc
        ).isoformat()
    ),

    "validation_status": "PASSED",

    "artifacts": {
        name: str(path)
        for name, path in PERSISTED_ARTIFACTS.items()
    }
}

with open(
    MANIFEST_PATH,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        manifest,
        f,
        indent=2
    )

if not MANIFEST_PATH.exists():
    raise RuntimeError(
        "Notebook 05 manifest was not created."
    )

# ------------------------------------------------------------
# Final persistence report
# ------------------------------------------------------------

print("\n" + "-" * 100)
print("PERSISTED NOTEBOOK 05 ARTIFACTS")
print("-" * 100)

for name, path in PERSISTED_ARTIFACTS.items():

    print(
        f"{name:30s}: {path}"
    )

print(
    f"{'notebook_05_manifest':30s}: "
    f"{MANIFEST_PATH}"
)

print("\n" + "-" * 100)
print("PERSISTENCE VALIDATION")
print("-" * 100)

print(
    "Context persistence        : PASSED"
)

print(
    "Candidate pool persistence : PASSED"
)

print(
    "Recommendation persistence : PASSED"
)

print(
    "Top recommendation         : PASSED"
)

print(
    "Validation persistence     : PASSED"
)

print(
    "Coverage persistence       : PASSED"
)

print(
    "Profile persistence        : PASSED"
)

print(
    "Artifact verification      : PASSED"
)

print(
    "Manifest persistence       : PASSED"
)

print("\n" + "=" * 100)
print("NOTEBOOK 05.13 PERSISTENCE COMPLETE")
print("=" * 100)

print(
    f"Project root               : {PROJECT_ROOT}"
)

print(
    f"Notebook 05 directory      : {NOTEBOOK_05_DIR}"
)

print(
    f"Evaluation contexts        : {len(CONTEXT_DF)}"
)

print(
    f"Recommendation rows        : {len(RECOMMENDATION_DF)}"
)

print(
    f"Candidate strategies       : {len(CANDIDATE_STRATEGY_DF)}"
)

print(
    "STATUS: PASSED"
)

print("=" * 100)

AIR-LLM — NOTEBOOK 05.13
PERSISTING NOTEBOOK 05 OUTPUTS

----------------------------------------------------------------------------------------------------
PERSISTED NOTEBOOK 05 ARTIFACTS
----------------------------------------------------------------------------------------------------
llm_context                   : /content/drive/MyDrive/AIR_LLM_Research/data/notebook_05/contexts/llm_context.csv
llm_recommendations           : /content/drive/MyDrive/AIR_LLM_Research/data/notebook_05/recommendations/llm_recommendations.csv
top_recommendations           : /content/drive/MyDrive/AIR_LLM_Research/data/notebook_05/recommendations/top_recommendations.csv
candidate_strategy_pool       : /content/drive/MyDrive/AIR_LLM_Research/data/notebook_05/metadata/candidate_strategy_pool.csv
recommendation_validation     : /content/drive/MyDrive/AIR_LLM_Research/data/notebook_05/validation/recommendation_validation.csv
recommendation_coverage       : /content/drive/MyDrive/AIR_LLM_Research/data/note

In [17]:
# ============================================================
# NOTEBOOK 05.14 — DRIVE PERSISTENCE VALIDATION
# ============================================================

print("=" * 100)
print("AIR-LLM — NOTEBOOK 05.14")
print("VALIDATING DRIVE PERSISTENCE")
print("=" * 100)


PERSISTENCE_FILES = [

    CONTEXT_DIR
    / "llm_context.csv",

    CONTEXT_DIR
    / "recommendation_profile.csv",

    METADATA_DIR
    / "candidate_strategy_pool.csv",

    RECOMMENDATION_DIR
    / "llm_recommendations.csv",

    RECOMMENDATION_DIR
    / "top_recommendations.csv",

    VALIDATION_DIR
    / "recommendation_validation.csv",

    VALIDATION_DIR
    / "coverage_summary.csv",

    VALIDATION_DIR
    / "recommendation_coverage.csv"
]


missing_files = [
    str(path)
    for path in PERSISTENCE_FILES
    if not path.exists()
]


if missing_files:

    raise RuntimeError(
        "Drive persistence validation failed:\n"
        + "\n".join(
            missing_files
        )
    )


print(
    f"Files validated : "
    f"{len(PERSISTENCE_FILES)}"
)

print(
    "Drive persistence : PASSED"
)

AIR-LLM — NOTEBOOK 05.14
VALIDATING DRIVE PERSISTENCE
Files validated : 8
Drive persistence : PASSED


In [18]:
# ============================================================
# NOTEBOOK 05.15 — NOTEBOOK MANIFEST
# ============================================================

print("=" * 100)
print("AIR-LLM — NOTEBOOK 05.15")
print("CREATING NOTEBOOK MANIFEST")
print("=" * 100)


manifest = {

    "notebook":
        "05_LLM_Recommendation_Engine",

    "version":
        RECOMMENDATION_VERSION,

    "seed":
        SEED,

    "datasets":
        DATASETS,

    "target_registry":
        TARGET_REGISTRY,

    "technical_features_excluded":
        list(
            TECHNICAL_FEATURES
        ),

    "profiled_features":
        int(
            len(FEATURE_PROFILE_DF)
        ),

    "incomplete_features":
        int(
            len(CONTEXT_DF)
        ),

    "candidate_strategies":
        int(
            len(CANDIDATE_STRATEGY_DF)
        ),

    "top_k":
        TOP_K,

    "recommendation_rows":
        int(
            len(RECOMMENDATION_DF)
        ),

    "coverage_rate":
        float(
            overall_coverage
        ),

    "recommendation_source":
        RECOMMENDATION_SOURCE,

    "outputs": {

        "context":
            str(
                CONTEXT_DIR
                / "llm_context.csv"
            ),

        "recommendations":
            str(
                RECOMMENDATION_DIR
                / "llm_recommendations.csv"
            ),

        "top_recommendations":
            str(
                RECOMMENDATION_DIR
                / "top_recommendations.csv"
            ),

        "candidate_pool":
            str(
                METADATA_DIR
                / "candidate_strategy_pool.csv"
            ),

        "validation":
            str(
                VALIDATION_DIR
                / "recommendation_validation.csv"
            ),

        "coverage":
            str(
                VALIDATION_DIR
                / "recommendation_coverage.csv"
            )
    }
}


MANIFEST_PATH = (
    METADATA_DIR
    / "notebook_05_manifest.json"
)


with open(
    MANIFEST_PATH,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        manifest,
        f,
        indent=2
    )


print(
    f"Manifest saved:\n"
    f"{MANIFEST_PATH}"
)

AIR-LLM — NOTEBOOK 05.15
CREATING NOTEBOOK MANIFEST
Manifest saved:
/content/drive/MyDrive/AIR_LLM_Research/data/notebook_05/metadata/notebook_05_manifest.json


In [19]:
# ============================================================
# NOTEBOOK 05.16 — FINAL STATUS
# ============================================================

print("=" * 100)
print("AIR-LLM — NOTEBOOK 05 COMPLETE")
print("LLM RECOMMENDATION ENGINE")
print("=" * 100)


print("\nEXPERIMENTAL DESIGN")
print("-" * 100)

print(
    f"Datasets                    : "
    f"{len(DATASETS)}"
)

print(
    f"Profiled features           : "
    f"{len(FEATURE_PROFILE_DF)}"
)

print(
    f"Incomplete features         : "
    f"{len(CONTEXT_DF)}"
)

print(
    f"Candidate strategies        : "
    f"{len(CANDIDATE_STRATEGY_DF)}"
)

print(
    f"Candidates per feature      : "
    f"{TOP_K}"
)

print(
    f"Recommendation rows         : "
    f"{len(RECOMMENDATION_DF)}"
)


print("\nVALIDATION")
print("-" * 100)

print(
    "Target exclusion            : PASSED"
)

print(
    "Technical-feature exclusion : PASSED"
)

print(
    "Semantic feature typing     : PASSED"
)

print(
    "Context validation          : PASSED"
)

print(
    "Candidate pool validation   : PASSED"
)

print(
    "Recommendation validation   : PASSED"
)

print(
    "Ranking validation          : PASSED"
)

print(
    "Coverage validation         : PASSED"
)

print(
    "Drive persistence           : PASSED"
)


print("\nOUTPUTS")
print("-" * 100)

print(
    "Context :",
    CONTEXT_DIR / "llm_context.csv"
)

print(
    "Recommendations :",
    RECOMMENDATION_DIR
    / "llm_recommendations.csv"
)

print(
    "Top recommendations :",
    RECOMMENDATION_DIR
    / "top_recommendations.csv"
)

print(
    "Candidate pool :",
    METADATA_DIR
    / "candidate_strategy_pool.csv"
)

print(
    "Validation :",
    VALIDATION_DIR
    / "recommendation_validation.csv"
)

print(
    "Coverage :",
    VALIDATION_DIR
    / "recommendation_coverage.csv"
)

print(
    "Manifest :",
    MANIFEST_PATH
)


print("\n" + "=" * 100)
print(
    "ALL NOTEBOOK 05 VALIDATIONS PASSED"
)
print(
    "AIR-LLM LLM-GUIDED RECOMMENDATIONS ARE READY"
)
print(
    "NOTEBOOK 06 — CANDIDATE STRATEGY EVALUATION MAY BEGIN"
)
print("=" * 100)

AIR-LLM — NOTEBOOK 05 COMPLETE
LLM RECOMMENDATION ENGINE

EXPERIMENTAL DESIGN
----------------------------------------------------------------------------------------------------
Datasets                    : 3
Profiled features           : 80
Incomplete features         : 12
Candidate strategies        : 13
Candidates per feature      : 5
Recommendation rows         : 55

VALIDATION
----------------------------------------------------------------------------------------------------
Target exclusion            : PASSED
Technical-feature exclusion : PASSED
Semantic feature typing     : PASSED
Context validation          : PASSED
Candidate pool validation   : PASSED
Recommendation validation   : PASSED
Ranking validation          : PASSED
Coverage validation         : PASSED
Drive persistence           : PASSED

OUTPUTS
----------------------------------------------------------------------------------------------------
Context : /content/drive/MyDrive/AIR_LLM_Research/data/notebook_05/co